In [1]:
# Cell 1: نصب پکیج‌های مورد نیاز
!pip install -q transformer-lens torch numpy accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.8 MB/s eta 0:00:00


In [2]:
"""
m9_1_predictive_self_model.py
==============================
Milestone 9.1: Predictive Counterfactual Self-Model Benchmark.

Core Hypothesis:
  Can a lightweight epistemic self-model predict the exact behavioral effect (ΔMargin)
  of internal counterfactual interventions (Heads, MLPs, Residuals) on UNSEEN OOD tasks
  significantly better than empirical baselines (Global Mean, Component Mean, k-NN)?
"""

import torch
import numpy as np
import json
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. INTERVENTION DEFINITION & DYNAMICS SELF-MODEL
# ==============================================================================

@dataclass
class InterventionAction:
    action_id: str
    target_type: str  # "HEAD", "MLP", "RESID"
    layer: int
    head: int
    param: float

    def get_feature_vector(self) -> np.ndarray:
        type_code = 1.0 if self.target_type == "HEAD" else (2.0 if self.target_type == "MLP" else 3.0)
        return np.array([type_code, float(self.layer) / 28.0, float(self.head) / 16.0, self.param], dtype=np.float32)


class PredictiveSelfModelEngine:
    def __init__(self):
        self.experience_buffer: List[Dict] = []
        self.w_state = None
        self.bias = 0.0

    def record_experience(self, state_feat: np.ndarray, action: InterventionAction, actual_delta: float):
        act_feat = action.get_feature_vector()
        joint_feat = np.concatenate([state_feat, act_feat])
        self.experience_buffer.append({
            "joint_feat": joint_feat,
            "state_feat": state_feat,
            "action_id": action.action_id,
            "component": f"{action.target_type}_L{action.layer}H{action.head}",
            "delta": actual_delta
        })

    def fit_dynamics(self):
        """Fits an Epistemic Ridge Dynamics model on accumulated calibration experience."""
        if len(self.experience_buffer) < 4:
            return

        X = np.stack([e["joint_feat"] for e in self.experience_buffer])
        y = np.array([e["delta"] for e in self.experience_buffer])

        # Regularized Linear Ridge Dynamics (Fast, stable, analytical)
        lambda_reg = 0.10
        d = X.shape[1]
        self.w_state = np.linalg.solve(X.T @ X + lambda_reg * np.eye(d), X.T @ y)

    def predict_delta(self, state_feat: np.ndarray, action: InterventionAction) -> float:
        """Predicts counterfactual outcome using the fitted internal model."""
        if self.w_state is None:
            return 0.0
        joint_feat = np.concatenate([state_feat, action.get_feature_vector()])
        pred = float(np.dot(joint_feat, self.w_state))
        return pred

    # --- BASELINES ---
    def baseline_global_mean(self) -> float:
        if not self.experience_buffer: return 0.0
        return float(np.mean([e["delta"] for e in self.experience_buffer]))

    def baseline_component_mean(self, action: InterventionAction) -> float:
        comp_key = f"{action.target_type}_L{action.layer}H{action.head}"
        matches = [e["delta"] for e in self.experience_buffer if e["component"] == comp_key]
        return float(np.mean(matches)) if matches else self.baseline_global_mean()

    def baseline_knn(self, state_feat: np.ndarray, k: int = 2) -> float:
        if not self.experience_buffer: return 0.0
        dists = [
            (float(np.linalg.norm(state_feat - e["state_feat"])), e["delta"])
            for e in self.experience_buffer
        ]
        dists.sort(key=lambda x: x[0])
        top_k = [d[1] for d in dists[:min(k, len(dists))]]
        return float(np.mean(top_k))


# ==============================================================================
# 2. MECHANISTIC HARNESS
# ==============================================================================

class M9_1_Harness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.d_model = self.model.cfg.d_model
        self.d_head = self.model.cfg.d_head

    @torch.no_grad()
    def extract_state_feature(self, prompt: str) -> Tuple[int, int, float, np.ndarray]:
        tokens = self.model.to_tokens(prompt)
        nom_logits, cache = self.model.run_with_cache(tokens)
        last_logits = nom_logits[0, -1, :]

        top_tokens = torch.topk(last_logits, k=2).indices
        t1, t2 = int(top_tokens[0].item()), int(top_tokens[1].item())
        nom_margin = float((last_logits[t1] - last_logits[t2]).item())

        # Extract intrinsic state: DLA of L21H6, L21H1, MLP20 norm, Residual norm
        d_u = (self.model.W_U[:, t1] - self.model.W_U[:, t2]).detach().cpu().to(torch.float32)

        # L21H6 DLA
        z_21_6 = cache["blocks.21.attn.hook_z"][0, -1, 6, :].detach().cpu().to(torch.float32)
        w_o_21_6 = self.model.blocks[21].attn.W_O[6].detach().cpu().to(torch.float32)
        dla_21_6 = float(torch.dot(torch.matmul(z_21_6, w_o_21_6), d_u).item())

        # L21H1 DLA
        z_21_1 = cache["blocks.21.attn.hook_z"][0, -1, 1, :].detach().cpu().to(torch.float32)
        w_o_21_1 = self.model.blocks[21].attn.W_O[1].detach().cpu().to(torch.float32)
        dla_21_1 = float(torch.dot(torch.matmul(z_21_1, w_o_21_1), d_u).item())

        # L20 MLP norm
        mlp_20_out = cache["blocks.20.hook_mlp_out"][0, -1, :].detach().cpu().to(torch.float32)
        mlp_norm = float(mlp_20_out.norm().item())

        # Residual norm L21
        resid_21 = cache["blocks.21.hook_resid_pre"][0, -1, :].detach().cpu().to(torch.float32)
        resid_norm = float(resid_21.norm().item())

        state_feat = np.array([nom_margin, dla_21_6, dla_21_1, mlp_norm / 50.0, resid_norm / 100.0], dtype=np.float32)
        return t1, t2, nom_margin, state_feat

    @torch.no_grad()
    def execute_intervention(self, prompt: str, t1: int, t2: int, nom_margin: float, action: InterventionAction) -> float:
        tokens = self.model.to_tokens(prompt)

        # Hook definition based on intervention target
        if action.target_type == "HEAD":
            def hook_fn(act, hook):
                act[:, -1, action.head, :] *= action.param
                return act
            hook_point = f"blocks.{action.layer}.attn.hook_z"
        elif action.target_type == "MLP":
            def hook_fn(act, hook):
                act[:, -1, :] *= action.param
                return act
            hook_point = f"blocks.{action.layer}.hook_mlp_out"
        elif action.target_type == "RESID":
            def hook_fn(act, hook):
                act[:, -1, :] += action.param * (act[:, -1, :] / (act[:, -1, :].norm() + 1e-8))
                return act
            hook_point = f"blocks.{action.layer}.hook_resid_pre"
        else:
            raise ValueError(f"Unknown target: {action.target_type}")

        with self.model.hooks(fwd_hooks=[(hook_point, hook_fn)]):
            p_logits = self.model(tokens)

        p_margin = float((p_logits[0, -1, t1] - p_logits[0, -1, t2]).item())
        actual_delta = float(p_margin - nom_margin)  # Positive = intervention improved margin, Negative = degraded
        return actual_delta


# ==============================================================================
# 3. CORPUS & INTERVENTION ACTIONS
# ==============================================================================

def get_calibration_corpus():
    return [
        {"id": "CAL_01", "prompt": "The capital of France is the city of"},
        {"id": "CAL_02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "CAL_03", "prompt": "The official capital city of Germany is"},
        {"id": "CAL_04", "prompt": "The city serving as the capital of Spain is definitely"},
        {"id": "CAL_05", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "CAL_06", "prompt": "David passed the heavy book to Sarah. The book was taken by"}
    ]

def get_ood_evaluation_corpus():
    return [
        {"id": "OOD_ADV_01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "OOD_ADV_02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "OOD_ADV_03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "OOD_ADV_04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "OOD_ADV_05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "OOD_ADV_06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "OOD_ADV_07", "prompt": "The official capital of Germany is definitely not Rome, it is"},
        {"id": "OOD_ADV_08", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"}
    ]

def get_intervention_suite() -> List[InterventionAction]:
    return [
        InterventionAction("A1_L21H06_ScaleDown", "HEAD", 21, 6, 0.85),
        InterventionAction("A2_L21H06_ScaleUp",   "HEAD", 21, 6, 1.15),
        InterventionAction("A3_L21H01_Damp",       "HEAD", 21, 1, 0.80),
        InterventionAction("A4_L20MLP_Perturb",    "MLP",  20, 0, 0.90),
        InterventionAction("A5_L21Resid_Jitter",   "RESID", 21, 0, 0.50)
    ]


# ==============================================================================
# 4. MASTER BENCHMARK RUNNER
# ==============================================================================

def run_milestone_9_1():
    harness = M9_1_Harness()
    self_model = PredictiveSelfModelEngine()

    calib_tasks = get_calibration_corpus()
    ood_tasks = get_ood_evaluation_corpus()
    actions = get_intervention_suite()

    print("\n" + "=" * 135)
    print("📚 PHASE 1: SELF-MODEL CALIBRATION & EXPERIENCE ACCUMULATION (Training Dynamics)")
    print("=" * 135)

    for task in calib_tasks:
        t1, t2, nom_m, s_feat = harness.extract_state_feature(task["prompt"])
        for act in actions:
            actual_d = harness.execute_intervention(task["prompt"], t1, t2, nom_m, act)
            self_model.record_experience(s_feat, act, actual_d)

    self_model.fit_dynamics()
    print(f"✅ Dynamics model trained on {len(self_model.experience_buffer)} empirical calibration experiences.")

    # -------------------------------------------------------------------------
    # PHASE 2: STRICT OOD COUNTERFACTUAL PREDICTION EVALUATION
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("🔬 PHASE 2: OOD COUNTERFACTUAL INTERVENTION PREDICTION (Zero-Shot on Adversarial Tasks)")
    print("=" * 135)
    print(f"{'Task ID':<16} | {'Action':<22} | {'Actual Δ':<10} | {'Self-Model':<12} | {'Comp Mean':<12} | {'k-NN':<10} | {'Global Mean'}")
    print("-" * 135)

    errs_self_model = []
    errs_comp_mean = []
    errs_knn = []
    errs_global_mean = []
    sign_hits_self_model = 0
    total_evals = 0

    trace_records = []

    for task in ood_tasks:
        t1, t2, nom_m, s_feat = harness.extract_state_feature(task["prompt"])
        for act in actions:
            actual_d = harness.execute_intervention(task["prompt"], t1, t2, nom_m, act)

            # Model & Baseline predictions
            pred_self = self_model.predict_delta(s_feat, act)
            pred_comp = self_model.baseline_component_mean(act)
            pred_knn = self_model.baseline_knn(s_feat)
            pred_glob = self_model.baseline_global_mean()

            # Record Absolute Errors
            e_self = abs(pred_self - actual_d)
            e_comp = abs(pred_comp - actual_d)
            e_knn = abs(pred_knn - actual_d)
            e_glob = abs(pred_glob - actual_d)

            errs_self_model.append(e_self)
            errs_comp_mean.append(e_comp)
            errs_knn.append(e_knn)
            errs_global_mean.append(e_glob)

            # Check directional sign accuracy (sign match)
            if (pred_self * actual_d > 0) or (abs(actual_d) < 1e-4 and abs(pred_self) < 1e-4):
                sign_hits_self_model += 1
            total_evals += 1

            print(f"{task['id']:<16} | {act.action_id:<22} | {actual_d:+8.4f}   | {pred_self:+10.4f}   | {pred_comp:+10.4f}   | {pred_knn:+8.4f}   | {pred_glob:+10.4f}")

            trace_records.append({
                "task": task["id"],
                "action": act.action_id,
                "actual_delta": actual_d,
                "pred_self_model": pred_self,
                "pred_comp_mean": pred_comp,
                "pred_knn": pred_knn,
                "pred_global_mean": pred_glob
            })

    # -------------------------------------------------------------------------
    # STATISTICAL BENCHMARK SUMMARY
    # -------------------------------------------------------------------------
    mae_self = float(np.mean(errs_self_model))
    mae_comp = float(np.mean(errs_comp_mean))
    mae_knn = float(np.mean(errs_knn))
    mae_glob = float(np.mean(errs_global_mean))
    sign_acc = (sign_hits_self_model / total_evals) * 100.0

    print("\n" + "=" * 135)
    print("📊 PREDICTIVE SELF-MODEL ACCURACY VS EMPIRICAL BASELINES (OOD Tasks)")
    print("=" * 135)
    print(f"  • Predictive Self-Model MAE      : {mae_self:.4f}  (Sign Accuracy: {sign_acc:.1f}%)")
    print(f"  • Component Static Mean MAE      : {mae_comp:.4f}  [Relative Improvement: {((mae_comp - mae_self)/mae_comp)*100:+.1f}%]")
    print(f"  • State k-NN (k=2) MAE           : {mae_knn:.4f}  [Relative Improvement: {((mae_knn - mae_self)/mae_knn)*100:+.1f}%]")
    print(f"  • Global Mean MAE                : {mae_glob:.4f}  [Relative Improvement: {((mae_glob - mae_self)/mae_glob)*100:+.1f}%]")
    print("-" * 135)

    if mae_self < mae_comp and mae_self < mae_knn and sign_acc >= 65.0:
        print("👉 BENCHMARK VERDICT: 🟢 PREDICTIVE_SELF_MODEL_VALIDATED")
        print("   (Self-Model successfully generalizes counterfactual intervention dynamics across OOD regimes)")
    else:
        print("👉 BENCHMARK VERDICT: 🔴 INSUFFICIENT_PREDICTIVE_GENERALIZATION")
    print("=" * 135 + "\n")

    with open("m9_1_predictive_self_model_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "metrics": {
                "mae_self_model": mae_self,
                "mae_comp_mean": mae_comp,
                "mae_knn": mae_knn,
                "mae_global_mean": mae_glob,
                "sign_accuracy_pct": sign_acc
            },
            "eval_trace": trace_records
        }, f, indent=2, ensure_ascii=False)
    print("📁 Benchmark telemetry saved to: m9_1_predictive_self_model_results.json")


if __name__ == "__main__":
    run_milestone_9_1()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1721/177579978.py:108: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

📚 PHASE 1: SELF-MODEL CALIBRATION & EXPERIENCE ACCUMULATION (Training Dynamics)
✅ Dynamics model trained on 30 empirical calibration experiences.

🔬 PHASE 2: OOD COUNTERFACTUAL INTERVENTION PREDICTION (Zero-Shot on Adversarial Tasks)
Task ID          | Action                 | Actual Δ   | Self-Model   | Comp Mean    | k-NN       | Global Mean
---------------------------------------------------------------------------------------------------------------------------------------
OOD_ADV_01       | A1_L21H06_ScaleDown    |  +0.0000   |    +0.1285   |    +0.0000   |  +0.0625   |    -0.0042
OOD_ADV_01       | A2_L21H06_ScaleUp      |  -0.2500   |    +0.1051   |    +0.0000   |  +0.0625   |    -0.0042
OOD_ADV_01       | A3_L21H01_Damp         |  +0.1250   |    +0.1348   |    +0.0208   |  +0.0625   |    -0.0042
OOD_ADV_01       | A4_L20MLP_Perturb      |  +0.0000   |    +0.0946   |    -0.0208   |  +0.0625   |    -0.0042
OOD_ADV_

In [3]:
"""
m9_2_active_fingerprint_scaling.py
===================================
Milestone 9.2: Micro-Probe Perturbation Fingerprint to Macro-Intervention Transfer.

Core Hypothesis:
  A low-cost active perturbation fingerprint Phi(x) = [P1, P2, P3, P4] predicts
  the consequence of heavy, unseen macro-interventions on OOD task families
  significantly better than passive residual states or static component averages.
"""

import torch
import numpy as np
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. ACTION & FINGERPRINT STRUCTURES
# ==============================================================================

@dataclass
class MacroIntervention:
    action_id: str
    layer: int
    head: int
    scale: float
    desc: str


class ActiveFingerprintSelfModel:
    def __init__(self):
        self.train_fingerprints = []
        self.train_passive_states = []
        self.train_macro_deltas = {a_id: [] for a_id in ["A1_ZeroAblate", "A2_HeavyBoost", "A3_CrossHeadSuppress"]}
        self.w_fingerprint = {}
        self.w_passive = {}

    def record_training_instance(self, phi: np.ndarray, passive_s: np.ndarray, macro_deltas: Dict[str, float]):
        self.train_fingerprints.append(phi)
        self.train_passive_states.append(passive_s)
        for a_id, d in macro_deltas.items():
            self.train_macro_deltas[a_id].append(d)

    def fit_transfer_models(self):
        X_phi = np.stack(self.train_fingerprints)
        X_pass = np.stack(self.train_passive_states)

        lambda_reg = 0.05
        d_phi, d_pass = X_phi.shape[1], X_pass.shape[1]

        for a_id, y_list in self.train_macro_deltas.items():
            y = np.array(y_list)
            # Ridge fit for Active Fingerprint
            self.w_fingerprint[a_id] = np.linalg.solve(X_phi.T @ X_phi + lambda_reg * np.eye(d_phi), X_phi.T @ y)
            # Ridge fit for Passive State (Baseline)
            self.w_passive[a_id] = np.linalg.solve(X_pass.T @ X_pass + lambda_reg * np.eye(d_pass), X_pass.T @ y)

    def predict_active(self, phi: np.ndarray, a_id: str) -> float:
        return float(np.dot(phi, self.w_fingerprint[a_id]))

    def predict_passive(self, passive_s: np.ndarray, a_id: str) -> float:
        return float(np.dot(passive_s, self.w_passive[a_id]))

    def predict_mean_baseline(self, a_id: str) -> float:
        return float(np.mean(self.train_macro_deltas[a_id]))


# ==============================================================================
# 2. MECHANISTIC HARNESS
# ==============================================================================

class M9_2_Harness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.target_layer = 21
        self.target_head = 6

        # Orthogonal reference vector for micro-probe P3
        v_rand = torch.randn(self.model.cfg.d_head, dtype=torch.float32)
        self.v_ortho = (v_rand / v_rand.norm()).to(self.device)

    @torch.no_grad()
    def get_base_and_passive_state(self, prompt: str) -> Tuple[int, int, float, np.ndarray]:
        tokens = self.model.to_tokens(prompt)
        logits, cache = self.model.run_with_cache(tokens)
        last_logits = logits[0, -1, :]
        top_ids = torch.topk(last_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((last_logits[t1] - last_logits[t2]).item())

        # Passive residual feature vector (L21 resid_pre normalized)
        resid_21 = cache["blocks.21.hook_resid_pre"][0, -1, :].detach().cpu().to(torch.float32).numpy()
        passive_feat = resid_21 / (np.linalg.norm(resid_21) + 1e-8)
        return t1, t2, base_margin, passive_feat

    @torch.no_grad()
    def extract_micro_fingerprint(self, prompt: str, t1: int, t2: int, base_margin: float) -> np.ndarray:
        tokens = self.model.to_tokens(prompt)
        l, h = self.target_layer, self.target_head

        # P1: Scale-down (0.90)
        def p1_hook(act, hook): act[:, -1, h, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[(f"blocks.{l}.attn.hook_z", p1_hook)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P2: Scale-up (1.10)
        def p2_hook(act, hook): act[:, -1, h, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[(f"blocks.{l}.attn.hook_z", p2_hook)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P3: Orthogonal Jitter (+0.08)
        def p3_hook(act, hook): act[:, -1, h, :] += 0.08 * act[:, -1, h, :].norm() * self.v_ortho.to(act.dtype); return act
        with self.model.hooks(fwd_hooks=[(f"blocks.{l}.attn.hook_z", p3_hook)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P4: Damping (-0.08)
        def p4_hook(act, hook): act[:, -1, h, :] -= 0.08 * act[:, -1, h, :]; return act
        with self.model.hooks(fwd_hooks=[(f"blocks.{l}.attn.hook_z", p4_hook)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        return np.array([d1, d2, d3, d4, d1 + d2, abs(d3)], dtype=np.float32)

    @torch.no_grad()
    def execute_macro_intervention(self, prompt: str, t1: int, t2: int, base_margin: float, action: MacroIntervention) -> float:
        tokens = self.model.to_tokens(prompt)
        l, h = action.layer, action.head

        def macro_hook(act, hook):
            act[:, -1, h, :] *= action.scale
            return act

        with self.model.hooks(fwd_hooks=[(f"blocks.{l}.attn.hook_z", macro_hook)]):
            macro_logits = self.model(tokens)

        macro_margin = float((macro_logits[0, -1, t1] - macro_logits[0, -1, t2]).item())
        return float(macro_margin - base_margin)


# ==============================================================================
# 3. CORPUS (STRICT TRAIN / TEST REGIME SEPARATION)
# ==============================================================================

def get_train_calibration_tasks():
    return [
        # Family 1: Clean Facts
        {"id": "TR_F01", "prompt": "The capital of France is the city of"},
        {"id": "TR_F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "TR_F03", "prompt": "The official capital city of Germany is"},
        {"id": "TR_F04", "prompt": "The historic capital of Italy is the city of"},
        # Family 2: Direct In-Context Bindings
        {"id": "TR_B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "TR_B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "TR_B03", "prompt": "John sent a private email to Mary. The email reached"},
        # Family 3: Explicit Negation
        {"id": "TR_N01", "prompt": "The capital of Spain is definitely not Rome, it is"},
        {"id": "TR_N02", "prompt": "The language spoken in Brazil is not Spanish, it is"}
    ]

def get_test_ood_tasks():
    return [
        # Family 4: High-Conflict Distractors
        {"id": "TE_D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "TE_D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "TE_D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "TE_D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        # Family 5: Syntactic / Interference Traps
        {"id": "TE_I01", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "TE_I02", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "TE_I03", "prompt": "Alice gave a book to Bob, but Charlie immediately seized it. The book is with"},
        {"id": "TE_I04", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"}
    ]

def get_macro_actions() -> List[MacroIntervention]:
    return [
        MacroIntervention("A1_ZeroAblate", 21, 6, 0.0, "Complete zero ablation of L21H06"),
        MacroIntervention("A2_HeavyBoost", 21, 6, 2.0, "Heavy 2x scaling boost of L21H06"),
        MacroIntervention("A3_CrossHeadSuppress", 21, 1, 0.1, "Near-zero suppression of structural L21H01")
    ]


# ==============================================================================
# 4. MASTER EXPERIMENTAL CONTROLLER
# ==============================================================================

def run_milestone_9_2():
    harness = M9_2_Harness()
    self_model = ActiveFingerprintSelfModel()

    train_tasks = get_train_calibration_tasks()
    test_tasks = get_test_ood_tasks()
    macro_actions = get_macro_actions()

    print("\n" + "=" * 135)
    print("📚 PHASE 1: COLLECTING ACTIVE MICRO-FINGERPRINTS ON TRAINING REGIMES")
    print("=" * 135)

    for item in train_tasks:
        t1, t2, nom_m, pass_s = harness.get_base_and_passive_state(item["prompt"])
        phi = harness.extract_micro_fingerprint(item["prompt"], t1, t2, nom_m)

        macro_deltas = {}
        for act in macro_actions:
            d_macro = harness.execute_macro_intervention(item["prompt"], t1, t2, nom_m, act)
            macro_deltas[act.action_id] = d_macro

        self_model.record_training_instance(phi, pass_s, macro_deltas)

    self_model.fit_transfer_models()
    print(f"✅ Trained Active Transfer Maps across {len(train_tasks)} calibration instances.")

    # -------------------------------------------------------------------------
    # PHASE 2: OOD EVALUATION (MICRO-FINGERPRINT VS BASELINES)
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("🔬 PHASE 2: EVALUATION OF TRANSFER DYNAMICS ON UNSEEN OOD REGIMES")
    print("=" * 135)
    print(f"{'Task ID':<12} | {'Macro Action':<20} | {'Actual Δ':<10} | {'Active Φ(x)':<12} | {'Passive State':<14} | {'Global Mean'}")
    print("-" * 135)

    errs_active, errs_passive, errs_mean = [], [], []
    actual_all, pred_active_all, pred_passive_all = [], [], []

    eval_records = []

    for item in test_tasks:
        t1, t2, nom_m, pass_s = harness.get_base_and_passive_state(item["prompt"])
        phi = harness.extract_micro_fingerprint(item["prompt"], t1, t2, nom_m)

        for act in macro_actions:
            actual_d = harness.execute_macro_intervention(item["prompt"], t1, t2, nom_m, act)

            p_active = self_model.predict_active(phi, act.action_id)
            p_passive = self_model.predict_passive(pass_s, act.action_id)
            p_mean = self_model.predict_mean_baseline(act.action_id)

            errs_active.append(abs(p_active - actual_d))
            errs_passive.append(abs(p_passive - actual_d))
            errs_mean.append(abs(p_mean - actual_d))

            actual_all.append(actual_d)
            pred_active_all.append(p_active)
            pred_passive_all.append(p_passive)

            print(f"{item['id']:<12} | {act.action_id:<20} | {actual_d:+8.4f}   | {p_active:+10.4f}   | {p_passive:+12.4f}   | {p_mean:+10.4f}")

            eval_records.append({
                "task": item["id"],
                "macro_action": act.action_id,
                "actual_delta": actual_d,
                "pred_active_fingerprint": p_active,
                "pred_passive_state": p_passive,
                "pred_global_mean": p_mean
            })

    # -------------------------------------------------------------------------
    # STATISTICAL TRANSFER AUDIT
    # -------------------------------------------------------------------------
    mae_active = float(np.mean(errs_active))
    mae_passive = float(np.mean(errs_passive))
    mae_mean = float(np.mean(errs_mean))

    # Pearson correlation on OOD tasks
    corr_active = float(np.corrcoef(actual_all, pred_active_all)[0, 1]) if np.std(pred_active_all) > 1e-6 else 0.0
    corr_passive = float(np.corrcoef(actual_all, pred_passive_all)[0, 1]) if np.std(pred_passive_all) > 1e-6 else 0.0

    print("\n" + "=" * 135)
    print("📊 ACTIVE PERTURBATION FINGERPRINT TRANSFER BENCHMARK SUMMARY")
    print("=" * 135)
    print(f"  • Active Micro-Fingerprint MAE   : {mae_active:.4f}  (Pearson ρ: {corr_active:+.4f})")
    print(f"  • Passive Residual State MAE     : {mae_passive:.4f}  (Pearson ρ: {corr_passive:+.4f}) [Rel. Diff: {((mae_passive-mae_active)/mae_passive)*100:+.1f}%]")
    print(f"  • Static Global Mean MAE         : {mae_mean:.4f}  [Rel. Diff: {((mae_mean-mae_active)/mae_mean)*100:+.1f}%]")
    print("-" * 135)

    if mae_active < mae_passive and corr_active >= 0.50:
        print("👉 BENCHMARK VERDICT: 🟢 ACTIVE_FINGERPRINT_TRANSFER_VALIDATED")
        print("   (Active micro-probes reliably transfer to macro-intervention outcomes across OOD task families)")
    else:
        print("👉 BENCHMARK VERDICT: 🔴 INSUFFICIENT_FINGERPRINT_TRANSFER")
    print("=" * 135 + "\n")

    with open("m9_2_active_fingerprint_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "metrics": {
                "mae_active": mae_active,
                "mae_passive": mae_passive,
                "mae_mean": mae_mean,
                "corr_active": corr_active,
                "corr_passive": corr_passive
            },
            "eval_records": eval_records
        }, f, indent=2, ensure_ascii=False)
    print("📁 Telemetry saved to: m9_2_active_fingerprint_results.json")


if __name__ == "__main__":
    run_milestone_9_2()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1721/1798353653.py:83: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

📚 PHASE 1: COLLECTING ACTIVE MICRO-FINGERPRINTS ON TRAINING REGIMES
✅ Trained Active Transfer Maps across 9 calibration instances.

🔬 PHASE 2: EVALUATION OF TRANSFER DYNAMICS ON UNSEEN OOD REGIMES
Task ID      | Macro Action         | Actual Δ   | Active Φ(x)  | Passive State  | Global Mean
---------------------------------------------------------------------------------------------------------------------------------------
TE_D01       | A1_ZeroAblate        |  +0.2500   |    -0.0016   |      +0.4377   |    -0.1250
TE_D01       | A2_HeavyBoost        |  -0.7500   |    -0.0920   |      +0.1173   |    -0.0972
TE_D01       | A3_CrossHeadSuppress |  +0.6250   |    +0.0964   |      +0.2469   |    +0.1667
TE_D02       | A1_ZeroAblate        |  +0.0000   |    +0.0000   |      -0.1640   |    -0.1250
TE_D02       | A2_HeavyBoost        |  +0.0000   |    +0.0000   |      +0.0132   |    -0.0972
TE_D02       | A3_CrossHeadSuppress 

In [4]:
"""
m9_3_longitudinal_regime_discovery.py
======================================
Milestone 9.3: Longitudinal Experience Accumulation & Regime-Conditioned Transfer.

Key Enhancements:
  1. Scaled Corpus (60 Tasks across 6 Distinct Semantic/Syntactic Families).
  2. Multi-Organ Micro-Fingerprinting (L21H06 + L21H01 + L20MLP).
  3. Two-Stage Dynamics:
     - Stage A: Unsupervised Causal Regime Discovery (K-Means on Fingerprints).
     - Stage B: Regime-Conditioned Local Linear Transfer to Heavy Macro-Interventions.
  4. Benchmark Arena: Compares Regime-Conditioned Model against Flat Ridge, k-NN Memory, and Global Mean.
"""

import torch
import numpy as np
import json
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. EPISODIC MEMORY & TWO-STAGE REGIME SELF-MODEL
# ==============================================================================

@dataclass
class EpisodeExperience:
    task_id: str
    family: str
    phi: np.ndarray          # Multi-organ micro-fingerprint
    macro_deltas: Dict[str, float]
    assigned_regime: int = -1


class LongitudinalSelfModel:
    def __init__(self, n_regimes: int = 3):
        self.n_regimes = n_regimes
        self.memory_bank: List[EpisodeExperience] = []
        self.regime_centroids = None
        self.regime_weights = {}     # {macro_action: {regime_id: w_vector}}
        self.flat_weights = {}       # {macro_action: w_vector}
        self.macro_keys = ["A1_ZeroAblate_H6", "A2_HeavyBoost_H6", "A3_Suppress_H1"]

    def store_episode(self, exp: EpisodeExperience):
        self.memory_bank.append(exp)

    def fit(self):
        """Discovers causal regimes via clustering, then fits regime-specific transfer maps."""
        X_phi = np.stack([e.phi for e in self.memory_bank])
        N, D = X_phi.shape

        # --- Stage A: Regime Discovery (K-Means Clustering on Fingerprint Space) ---
        np.random.seed(SEED)
        init_idx = np.random.choice(N, self.n_regimes, replace=False)
        self.regime_centroids = X_phi[init_idx].copy()

        # Simple iterative K-Means
        for _ in range(25):
            dists = np.linalg.norm(X_phi[:, None, :] - self.regime_centroids[None, :, :], axis=-1)
            labels = np.argmin(dists, axis=1)
            for k in range(self.n_regimes):
                members = X_phi[labels == k]
                if len(members) > 0:
                    self.regime_centroids[k] = members.mean(axis=0)

        # Assign regimes to training episodes
        dists = np.linalg.norm(X_phi[:, None, :] - self.regime_centroids[None, :, :], axis=-1)
        final_labels = np.argmin(dists, axis=1)
        for idx, e in enumerate(self.memory_bank):
            e.assigned_regime = int(final_labels[idx])

        # --- Stage B: Fit Transfer Models (Flat Ridge & Regime-Conditioned) ---
        lambda_reg = 0.05
        for a_id in self.macro_keys:
            y_all = np.array([e.macro_deltas[a_id] for e in self.memory_bank])

            # 1. Flat Linear Transfer (M9.2 Baseline with scaled data)
            self.flat_weights[a_id] = np.linalg.solve(
                X_phi.T @ X_phi + lambda_reg * np.eye(D), X_phi.T @ y_all
            )

            # 2. Regime-Conditioned Transfer
            self.regime_weights[a_id] = {}
            for k in range(self.n_regimes):
                mask = (final_labels == k)
                X_k = X_phi[mask]
                y_k = y_all[mask]
                if len(X_k) >= D:
                    w_k = np.linalg.solve(X_k.T @ X_k + lambda_reg * np.eye(D), X_k.T @ y_k)
                elif len(X_k) > 0:
                    w_k = np.linalg.solve(X_k.T @ X_k + 0.5 * np.eye(D), X_k.T @ y_k)
                else:
                    w_k = self.flat_weights[a_id].copy()
                self.regime_weights[a_id][k] = w_k

    def infer_regime(self, phi: np.ndarray) -> int:
        dists = np.linalg.norm(self.regime_centroids - phi[None, :], axis=-1)
        return int(np.argmin(dists))

    def predict_regime_conditioned(self, phi: np.ndarray, a_id: str) -> Tuple[float, int]:
        r_id = self.infer_regime(phi)
        pred = float(np.dot(phi, self.regime_weights[a_id][r_id]))
        return pred, r_id

    def predict_flat_linear(self, phi: np.ndarray, a_id: str) -> float:
        return float(np.dot(phi, self.flat_weights[a_id]))

    def predict_knn_memory(self, phi: np.ndarray, a_id: str, k: int = 3) -> float:
        dists = [
            (float(np.linalg.norm(phi - e.phi)), e.macro_deltas[a_id])
            for e in self.memory_bank
        ]
        dists.sort(key=lambda x: x[0])
        top_k = [d[1] for d in dists[:min(k, len(dists))]]
        return float(np.mean(top_k))

    def predict_global_mean(self, a_id: str) -> float:
        return float(np.mean([e.macro_deltas[a_id] for e in self.memory_bank]))


# ==============================================================================
# 2. MECHANISTIC HARNESS
# ==============================================================================

class LongitudinalHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.d_head = self.model.cfg.d_head

        v_rand = torch.randn(self.d_head, dtype=torch.float32)
        self.v_ortho = (v_rand / v_rand.norm()).to(self.device)

    @torch.no_grad()
    def extract_multi_organ_fingerprint(self, prompt: str) -> Tuple[int, int, float, np.ndarray]:
        tokens = self.model.to_tokens(prompt)
        nom_logits = self.model(tokens)
        last_logits = nom_logits[0, -1, :]
        top_ids = torch.topk(last_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((last_logits[t1] - last_logits[t2]).item())

        # Micro-probes across L21H06, L21H01, and L20MLP
        # P1 on L21H06 (0.90 Scale)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d_p1_h6 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P2 on L21H06 (1.10 Scale)
        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d_p2_h6 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P3 on L21H06 (Ortho Jitter)
        def p3_h6(act, hook): act[:, -1, 6, :] += 0.08 * act[:, -1, 6, :].norm() * self.v_ortho.to(act.dtype); return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h6)]):
            d_p3_h6 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P1 on L21H01 (0.90 Scale)
        def p1_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h1)]):
            d_p1_h1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P1 on L20 MLP (0.90 Scale)
        def p1_mlp(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p1_mlp)]):
            d_p1_mlp = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi = np.array([
            d_p1_h6, d_p2_h6, d_p3_h6, d_p1_h1, d_p1_mlp,
            d_p1_h6 + d_p2_h6, abs(d_p3_h6), base_margin / 5.0
        ], dtype=np.float32)

        return t1, t2, base_margin, phi

    @torch.no_grad()
    def measure_macro_interventions(self, prompt: str, t1: int, t2: int, base_margin: float) -> Dict[str, float]:
        tokens = self.model.to_tokens(prompt)
        results = {}

        # Macro 1: Zero Ablation of L21H06
        def m1_hook(act, hook): act[:, -1, 6, :] *= 0.0; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", m1_hook)]):
            results["A1_ZeroAblate_H6"] = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # Macro 2: Heavy Boost of L21H06 (2.0x)
        def m2_hook(act, hook): act[:, -1, 6, :] *= 2.0; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", m2_hook)]):
            results["A2_HeavyBoost_H6"] = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # Macro 3: Suppression of Structural L21H01 (0.1x)
        def m3_hook(act, hook): act[:, -1, 1, :] *= 0.1; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", m3_hook)]):
            results["A3_Suppress_H1"] = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        return results


# ==============================================================================
# 3. PROCEDURAL MULTI-FAMILY CORPUS (60 DISTINCT TASKS)
# ==============================================================================

def generate_longitudinal_corpus() -> Tuple[List[Dict], List[Dict]]:
    # Training Families (36 Instances across 3 Families)
    train_corpus = []

    # 1. Clean Factual Statements (12 tasks)
    facts = [
        ("The capital of France is the city of", "FAM_FACT"),
        ("The capital of Japan is the bustling city of", "FAM_FACT"),
        ("The official capital city of Germany is", "FAM_FACT"),
        ("The historic capital of Italy is the city of", "FAM_FACT"),
        ("The capital of Spain is the city of", "FAM_FACT"),
        ("The capital of England is the city of", "FAM_FACT"),
        ("The capital of Russia is the city of", "FAM_FACT"),
        ("The capital of Egypt is the city of", "FAM_FACT"),
        ("The capital of Greece is the city of", "FAM_FACT"),
        ("The capital of China is the city of", "FAM_FACT"),
        ("The capital of Canada is the city of", "FAM_FACT"),
        ("The capital of Australia is the city of", "FAM_FACT")
    ]
    for idx, (p, f) in enumerate(facts, start=1):
        train_corpus.append({"id": f"TR_FACT_{idx:02d}", "family": f, "prompt": p})

    # 2. In-Context Entity Bindings (12 tasks)
    bindings = [
        ("Alice handed a red apple to Bob. The apple was received by", "FAM_BIND"),
        ("David passed the heavy book to Sarah. The book was taken by", "FAM_BIND"),
        ("John sent a private email to Mary. The email reached", "FAM_BIND"),
        ("Michael offered a cup of tea to Emma. The tea was accepted by", "FAM_BIND"),
        ("Tom tossed the baseball to Jack. The ball was caught by", "FAM_BIND"),
        ("Robert wrote a letter to Lisa. The letter was delivered to", "FAM_BIND"),
        ("William awarded a shiny medal to Oliver. The medal went to", "FAM_BIND"),
        ("James threw the yellow frisbee to Lucas. The frisbee was caught by", "FAM_BIND"),
        ("Daniel gave a fresh sandwich to Mason. The sandwich was eaten by", "FAM_BIND"),
        ("Henry presented an award to Ethan. The award was received by", "FAM_BIND"),
        ("Samuel mailed a package to Alexander. The package arrived for", "FAM_BIND"),
        ("George loaned some money to Noah. The money helped", "FAM_BIND")
    ]
    for idx, (p, f) in enumerate(bindings, start=1):
        train_corpus.append({"id": f"TR_BIND_{idx:02d}", "family": f, "prompt": p})

    # 3. Explicit Negation / Exclusion (12 tasks)
    negations = [
        ("The capital of Spain is definitely not Rome, it is", "FAM_NEG"),
        ("The official capital of Germany is not Munich, it is", "FAM_NEG"),
        ("The capital of France is not Marseille, it is", "FAM_NEG"),
        ("The capital of Japan is not Osaka, it is", "FAM_NEG"),
        ("The language spoken in Brazil is not Spanish, it is", "FAM_NEG"),
        ("The capital of Italy is not Milan, it is", "FAM_NEG"),
        ("The capital of Russia is not Saint Petersburg, it is", "FAM_NEG"),
        ("The currency of the United Kingdom is not the Euro, it is the", "FAM_NEG"),
        ("The capital of Canada is not Toronto, it is", "FAM_NEG"),
        ("The capital of Australia is not Sydney, it is", "FAM_NEG"),
        ("The capital of the United States is not New York, it is", "FAM_NEG"),
        ("The capital of China is not Shanghai, it is", "FAM_NEG")
    ]
    for idx, (p, f) in enumerate(negations, start=1):
        train_corpus.append({"id": f"TR_NEG_{idx:02d}", "family": f, "prompt": p})

    # OOD Evaluation Families (24 Instances across 3 Novel Families)
    test_corpus = []

    # 4. High-Conflict Distractors (8 tasks)
    distractors = [
        ("While Rome has the Colosseum, the Eiffel Tower stands in", "OOD_DISTRACT"),
        ("Even though Madrid is famous, the capital of Germany is", "OOD_DISTRACT"),
        ("Berlin has monuments, yet the ancient Colosseum stands in", "OOD_DISTRACT"),
        ("Although Tokyo is very large, the ancient Parthenon is located in", "OOD_DISTRACT"),
        ("Despite London being historic, the capital of Spain is definitely", "OOD_DISTRACT"),
        ("Paris is beautiful, but the capital of Italy is definitely", "OOD_DISTRACT"),
        ("Egypt has pyramids, but the Eiffel Tower was built in the city of", "OOD_DISTRACT"),
        ("Although Moscow has the Kremlin, the capital of France is the city of", "OOD_DISTRACT")
    ]
    for idx, (p, f) in enumerate(distractors, start=1):
        test_corpus.append({"id": f"TE_DIST_{idx:02d}", "family": f, "prompt": p})

    # 5. Syntactic Interference & Adversarial Ordering (8 tasks)
    interference = [
        ("Alice gave a book to Bob, but Charlie immediately seized it. The book is with", "OOD_INTERFERE"),
        ("David sent a letter to Sarah, yet Emma intercepted the envelope. The letter reached", "OOD_INTERFERE"),
        ("John passed a ball to Mary, but Lucas snatched it in midair. The ball belongs to", "OOD_INTERFERE"),
        ("Tom cooked dinner for Jack, but Oliver ate the entire plate. The food went to", "OOD_INTERFERE"),
        ("William wrote an article for Lisa, but Sophia published it under her name. The author is", "OOD_INTERFERE"),
        ("James offered a ride to Emma, but Alexander picked her up first. Emma rode with", "OOD_INTERFERE"),
        ("Daniel lent a camera to Mason, yet Ethan ended up using it all day. The camera was with", "OOD_INTERFERE"),
        ("Henry bought flowers for Mary, but Jack delivered them directly. Mary received flowers from", "OOD_INTERFERE")
    ]
    for idx, (p, f) in enumerate(interference, start=1):
        test_corpus.append({"id": f"TE_INT_{idx:02d}", "family": f, "prompt": p})

    # 6. Counterfactual / Flipped Contexts (8 tasks)
    flipped = [
        ("In an alternate universe where France relocated its capital to Lyon, the capital of France is", "OOD_FLIPPED"),
        ("Suppose Japan officially changed its capital from Tokyo to Kyoto. The official capital is", "OOD_FLIPPED"),
        ("If Germany renamed Munich as its federal capital city, the capital of Germany would be", "OOD_FLIPPED"),
        ("Imagine Italy established its seat of government in Venice. The capital of Italy is now", "OOD_FLIPPED"),
        ("In a world where Spain moved its government to Barcelona, the Spanish capital is", "OOD_FLIPPED"),
        ("Assuming the United Kingdom declared Edinburgh as its sole capital, the UK capital is", "OOD_FLIPPED"),
        ("If Russia designated Sochi as its primary capital city, the government of Russia resides in", "OOD_FLIPPED"),
        ("Suppose Egypt transferred all administrative headquarters to Alexandria. The capital city is", "OOD_FLIPPED")
    ]
    for idx, (p, f) in enumerate(flipped, start=1):
        test_corpus.append({"id": f"TE_FLIP_{idx:02d}", "family": f, "prompt": p})

    return train_corpus, test_corpus


# ==============================================================================
# 4. MASTER EXPERIMENTAL CONTROLLER
# ==============================================================================

def run_milestone_9_3():
    harness = LongitudinalHarness()
    self_model = LongitudinalSelfModel(n_regimes=3)
    train_tasks, test_tasks = generate_longitudinal_corpus()

    print("\n" + "=" * 135)
    print(f"📚 PHASE 1: LONGITUDINAL EXPERIENCE ACCUMULATION ({len(train_tasks)} Episodes across 3 Training Families)")
    print("=" * 135)

    for item in train_tasks:
        t1, t2, nom_m, phi = harness.extract_multi_organ_fingerprint(item["prompt"])
        macro_deltas = harness.measure_macro_interventions(item["prompt"], t1, t2, nom_m)

        episode = EpisodeExperience(
            task_id=item["id"],
            family=item["family"],
            phi=phi,
            macro_deltas=macro_deltas
        )
        self_model.store_episode(episode)

    self_model.fit()
    print(f"✅ Discovered {self_model.n_regimes} Causal Regimes and Fitted Regime-Conditioned Dynamics.")

    # Print Regime Distribution
    regime_counts = {}
    for e in self_model.memory_bank:
        regime_counts[e.assigned_regime] = regime_counts.get(e.assigned_regime, 0) + 1
    print("📊 Training Regime Membership Distribution:")
    for r_id, count in sorted(regime_counts.items()):
        print(f"  • Regime #{r_id}: {count} episodes ({count/len(train_tasks)*100:.1f}%)")

    # -------------------------------------------------------------------------
    # PHASE 2: OOD EVALUATION ACROSS NOVEL FAMILIES
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print(f"🔬 PHASE 2: EVALUATION ON UNSEEN OOD FAMILIES ({len(test_tasks)} Tasks across Distractors, Interference, Flipped)")
    print("=" * 135)
    print(f"{'Task ID':<14} | {'Action':<18} | {'Actual Δ':<9} | {'Regime-Model':<13} | {'Regime':<8} | {'Flat Ridge':<11} | {'k-NN Mem':<10} | {'Global Mean'}")
    print("-" * 135)

    errs_regime = []
    errs_flat = []
    errs_knn = []
    errs_mean = []

    actual_all = []
    pred_regime_all = []
    pred_flat_all = []

    trace_records = []

    for item in test_tasks:
        t1, t2, nom_m, phi = harness.extract_multi_organ_fingerprint(item["prompt"])
        actual_deltas = harness.measure_macro_interventions(item["prompt"], t1, t2, nom_m)

        for a_id in self_model.macro_keys:
            act_d = actual_deltas[a_id]
            p_regime, r_inferred = self_model.predict_regime_conditioned(phi, a_id)
            p_flat = self_model.predict_flat_linear(phi, a_id)
            p_knn = self_model.predict_knn_memory(phi, a_id, k=3)
            p_mean = self_model.predict_global_mean(a_id)

            errs_regime.append(abs(p_regime - act_d))
            errs_flat.append(abs(p_flat - act_d))
            errs_knn.append(abs(p_knn - act_d))
            errs_mean.append(abs(p_mean - act_d))

            actual_all.append(act_d)
            pred_regime_all.append(p_regime)
            pred_flat_all.append(p_flat)

            print(f"{item['id']:<14} | {a_id:<18} | {act_d:+7.4f}   | {p_regime:+10.4f}    | R#{r_inferred:<6} | {p_flat:+9.4f}   | {p_knn:+8.4f}   | {p_mean:+9.4f}")

            trace_records.append({
                "task": item["id"],
                "family": item["family"],
                "action": a_id,
                "actual_delta": act_d,
                "inferred_regime": r_inferred,
                "pred_regime_model": p_regime,
                "pred_flat_ridge": p_flat,
                "pred_knn_memory": p_knn,
                "pred_global_mean": p_mean
            })

    # -------------------------------------------------------------------------
    # STATISTICAL SUMMARY & AUDIT
    # -------------------------------------------------------------------------
    mae_regime = float(np.mean(errs_regime))
    mae_flat = float(np.mean(errs_flat))
    mae_knn = float(np.mean(errs_knn))
    mae_mean = float(np.mean(errs_mean))

    corr_regime = float(np.corrcoef(actual_all, pred_regime_all)[0, 1]) if np.std(pred_regime_all) > 1e-6 else 0.0
    corr_flat = float(np.corrcoef(actual_all, pred_flat_all)[0, 1]) if np.std(pred_flat_all) > 1e-6 else 0.0

    print("\n" + "=" * 135)
    print("📊 MILESTONE 9.3 SUMMARY: REGIME-CONDITIONED TRANSFER VS BASELINES (N=72 OOD Evaluations)")
    print("=" * 135)
    print(f"  • Two-Stage Regime Self-Model MAE : {mae_regime:.4f}  (Pearson ρ: {corr_regime:+.4f})")
    print(f"  • Flat Linear Ridge MAE           : {mae_flat:.4f}  (Pearson ρ: {corr_flat:+.4f}) [Advantage: {((mae_flat-mae_regime)/mae_flat)*100:+.1f}%]")
    print(f"  • Episodic k-NN Memory (k=3) MAE   : {mae_knn:.4f}  [Advantage: {((mae_knn-mae_regime)/mae_knn)*100:+.1f}%]")
    print(f"  • Static Global Mean MAE          : {mae_mean:.4f}  [Advantage: {((mae_mean-mae_regime)/mae_mean)*100:+.1f}%]")
    print("-" * 135)

    if mae_regime < mae_flat and mae_regime < mae_mean and corr_regime >= 0.40:
        print("👉 BENCHMARK VERDICT: 🟢 REGIME_CONDITIONED_TRANSFER_VALIDATED")
        print("   (Longitudinal experience accumulation and regime discovery successfully bridge micro-probes to macro consequences)")
    else:
        print("👉 BENCHMARK VERDICT: 🔴 INSUFFICIENT_REGIME_GENERALIZATION")
    print("=" * 135 + "\n")

    with open("m9_3_longitudinal_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "metrics": {
                "mae_regime_model": mae_regime,
                "mae_flat_ridge": mae_flat,
                "mae_knn_memory": mae_knn,
                "mae_global_mean": mae_mean,
                "corr_regime": corr_regime,
                "corr_flat": corr_flat
            },
            "eval_trace": trace_records
        }, f, indent=2, ensure_ascii=False)
    print("📁 Telemetry saved to: m9_3_longitudinal_results.json")


if __name__ == "__main__":
    run_milestone_9_3()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1721/431687953.py:134: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

📚 PHASE 1: LONGITUDINAL EXPERIENCE ACCUMULATION (36 Episodes across 3 Training Families)
✅ Discovered 3 Causal Regimes and Fitted Regime-Conditioned Dynamics.
📊 Training Regime Membership Distribution:
  • Regime #0: 7 episodes (19.4%)
  • Regime #1: 18 episodes (50.0%)
  • Regime #2: 11 episodes (30.6%)

🔬 PHASE 2: EVALUATION ON UNSEEN OOD FAMILIES (24 Tasks across Distractors, Interference, Flipped)
Task ID        | Action             | Actual Δ  | Regime-Model  | Regime   | Flat Ridge  | k-NN Mem   | Global Mean
---------------------------------------------------------------------------------------------------------------------------------------
TE_DIST_01     | A1_ZeroAblate_H6   | +0.2500   |    +0.0387    | R#2      |   +0.0087   |  -0.1250   |   +0.0382
TE_DIST_01     | A2_HeavyBoost_H6   | -0.7500   |    +0.0364    | R#2      |   -0.3360   |  +0.0833   |   -0.0694
TE_DIST_01     | A3_Suppress_H1     | +0.6250   |

In [5]:
"""
m9_3a_predictability_and_regime_audit.py
=========================================
Milestone 9.3A: Representation, Predictability & Regime Reality Audit.

Diagnostics:
  1. Permutation Significance Test: Real vs. 500 Shuffled-Phi Baselines (p-value).
  2. Cluster Causal Validity: Silhouette Score & ANOVA F-Test on Macro Responses.
  3. Readout Spectrum Analysis: Compares 5 candidate targets (Margin, Target Logit,
     Semantic Diff, KL Divergence, Probability Shift).
"""

import torch
import numpy as np
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC HARNESS & READOUT SPECTRUM
# ==============================================================================

class AuditHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.d_head = self.model.cfg.d_head
        v_rand = torch.randn(self.d_head, dtype=torch.float32)
        self.v_ortho = (v_rand / v_rand.norm()).to(self.device)

    @torch.no_grad()
    def extract_fingerprint_and_targets(self, item: Dict) -> Tuple[np.ndarray, Dict[str, Dict[str, float]]]:
        tokens = self.model.to_tokens(item["prompt"])
        nom_logits = self.model(tokens)
        last_logits = nom_logits[0, -1, :].detach().cpu().to(torch.float32)

        # Dynamic Top-1 / Top-2
        top_ids = torch.topk(last_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_dyn_margin = float((last_logits[t1] - last_logits[t2]).item())

        # Fixed Semantic Tokens
        c_id = int(self.model.to_single_token(item["correct"]))
        d_id = int(self.model.to_single_token(item["distractor"]))
        base_c_logit = float(last_logits[c_id].item())
        base_sem_margin = float((last_logits[c_id] - last_logits[d_id]).item())

        base_probs = torch.softmax(last_logits, dim=-1)
        base_c_prob = float(base_probs[c_id].item())

        # Micro-probes (L21H06, L21H01, L20MLP)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_dyn_margin

        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_dyn_margin

        def p3_h6(act, hook): act[:, -1, 6, :] += 0.08 * act[:, -1, 6, :].norm() * self.v_ortho.to(act.dtype); return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h6)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_dyn_margin

        def p1_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h1)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_dyn_margin

        def p1_mlp(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p1_mlp)]):
            d5 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_dyn_margin

        phi = np.array([d1, d2, d3, d4, d5, d1 + d2, abs(d3)], dtype=np.float32)

        # Macro Interventions Spectrum
        targets = {}
        macro_actions = {
            "A1_ZeroAblate": (21, 6, 0.0),
            "A2_HeavyBoost": (21, 6, 2.0)
        }

        for a_name, (l, h, sc) in macro_actions.items():
            def m_hook(act, hook): act[:, -1, h, :] *= sc; return act
            with self.model.hooks(fwd_hooks=[(f"blocks.{l}.attn.hook_z", m_hook)]):
                m_logits_t = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

            m_probs = torch.softmax(m_logits_t, dim=-1)

            # 5 Target Metrics:
            # 1. Dynamic Top-1/Top-2 Margin Delta
            delta_dyn_margin = float((m_logits_t[t1] - m_logits_t[t2]).item()) - base_dyn_margin
            # 2. Target Entity Logit Delta
            delta_target_logit = float(m_logits_t[c_id].item()) - base_c_logit
            # 3. Fixed Semantic Margin Delta (Correct - Distractor)
            delta_sem_margin = float((m_logits_t[c_id] - m_logits_t[d_id]).item()) - base_sem_margin
            # 4. KL Divergence (P_base || P_macro)
            kl_div = float(torch.sum(base_probs * (torch.log(base_probs + 1e-12) - torch.log(m_probs + 1e-12))).item())
            # 5. Probability Shift of Correct Token
            delta_prob = float(m_probs[c_id].item()) - base_c_prob

            targets[a_name] = {
                "dyn_margin": delta_dyn_margin,
                "target_logit": delta_target_logit,
                "sem_margin": delta_sem_margin,
                "kl_div": kl_div,
                "prob_shift": delta_prob
            }

        return phi, targets


# ==============================================================================
# 2. AUDIT CORPUS (40 RICH MULTI-FAMILY TASKS)
# ==============================================================================

def get_audit_corpus():
    tasks = [
        # Facts (10)
        {"id": "F01", "prompt": "The capital of France is the city of", "correct": " Paris", "distractor": " Rome"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of", "correct": " Tokyo", "distractor": " Kyoto"},
        {"id": "F03", "prompt": "The official capital city of Germany is", "correct": " Berlin", "distractor": " Munich"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of", "correct": " Rome", "distractor": " Milan"},
        {"id": "F05", "prompt": "The capital of Spain is the city of", "correct": " Madrid", "distractor": " Barcelona"},
        {"id": "F06", "prompt": "The capital of England is the city of", "correct": " London", "distractor": " York"},
        {"id": "F07", "prompt": "The capital of Russia is the city of", "correct": " Moscow", "distractor": " Paris"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of", "correct": " Cairo", "distractor": " Athens"},
        {"id": "F09", "prompt": "The capital of Greece is the city of", "correct": " Athens", "distractor": " Rome"},
        {"id": "F10", "prompt": "The capital of China is the city of", "correct": " Beijing", "distractor": " Shanghai"},
        # Bindings (10)
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by", "correct": " Bob", "distractor": " Alice"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by", "correct": " Sarah", "distractor": " David"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached", "correct": " Mary", "distractor": " John"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by", "correct": " Emma", "distractor": " Michael"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by", "correct": " Jack", "distractor": " Tom"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to", "correct": " Lisa", "distractor": " Robert"},
        {"id": "B07", "prompt": "William awarded a medal to Oliver. The medal went to", "correct": " Oliver", "distractor": " William"},
        {"id": "B08", "prompt": "James threw the frisbee to Lucas. The frisbee was caught by", "correct": " Lucas", "distractor": " James"},
        {"id": "B09", "prompt": "Daniel gave a sandwich to Mason. The sandwich was eaten by", "correct": " Mason", "distractor": " Daniel"},
        {"id": "B10", "prompt": "Henry presented a trophy to Ethan. The trophy was received by", "correct": " Ethan", "distractor": " Henry"},
        # Distractors / Interference (10)
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in", "correct": " Paris", "distractor": " Rome"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is", "correct": " Berlin", "distractor": " Madrid"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in", "correct": " Rome", "distractor": " Berlin"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in", "correct": " Athens", "distractor": " Tokyo"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely", "correct": " Madrid", "distractor": " London"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely", "correct": " Rome", "distractor": " Paris"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of", "correct": " Paris", "distractor": " Cairo"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of", "correct": " Paris", "distractor": " Moscow"},
        {"id": "D09", "prompt": "Alice gave a book to Bob, but Charlie immediately seized it. The book is with", "correct": " Charlie", "distractor": " Bob"},
        {"id": "D10", "prompt": "David sent a letter to Sarah, yet Emma intercepted the envelope. The letter reached", "correct": " Emma", "distractor": " Sarah"},
        # Negations (10)
        {"id": "N01", "prompt": "The capital of Spain is definitely not Rome, it is", "correct": " Madrid", "distractor": " Rome"},
        {"id": "N02", "prompt": "The official capital of Germany is not Munich, it is", "correct": " Berlin", "distractor": " Munich"},
        {"id": "N03", "prompt": "The capital of France is not Marseille, it is", "correct": " Paris", "distractor": " Rome"},
        {"id": "N04", "prompt": "The capital of Japan is not Osaka, it is", "correct": " Tokyo", "distractor": " Kyoto"},
        {"id": "N05", "prompt": "The capital of Italy is not Milan, it is", "correct": " Rome", "distractor": " Milan"},
        {"id": "N06", "prompt": "The capital of Russia is not Saint Petersburg, it is", "correct": " Moscow", "distractor": " Paris"},
        {"id": "N07", "prompt": "The currency of the United Kingdom is not the Euro, it is the", "correct": " Pound", "distractor": " Euro"},
        {"id": "N08", "prompt": "The capital of Canada is not Toronto, it is", "correct": " Ottawa", "distractor": " Toronto"},
        {"id": "N09", "prompt": "The capital of Australia is not Sydney, it is", "correct": " Canberra", "distractor": " Sydney"},
        {"id": "N10", "prompt": "The capital of China is not Shanghai, it is", "correct": " Beijing", "distractor": " Shanghai"}
    ]
    return tasks


# ==============================================================================
# 3. STATISTICAL AUDIT ENGINES (PURE NUMPY)
# ==============================================================================

def leave_one_out_ridge_mae(X: np.ndarray, y: np.ndarray, alpha: float = 0.1) -> float:
    N, D = X.shape
    errors = []
    for i in range(N):
        train_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_val, y_val = X[i:i+1], y[i]

        w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
        pred = float(np.dot(X_val, w)[0])
        errors.append(abs(pred - y_val))
    return float(np.mean(errors))


def compute_silhouette_and_anova(X: np.ndarray, y_macro: np.ndarray, k: int = 3) -> Tuple[float, float, float]:
    """Computes Silhouette score in Phi space and one-way ANOVA F-statistic on macro outcomes."""
    N = len(X)
    # Simple K-Means
    np.random.seed(SEED)
    centroids = X[np.random.choice(N, k, replace=False)].copy()
    for _ in range(20):
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=-1)
        labels = np.argmin(dists, axis=1)
        for c in range(k):
            members = X[labels == c]
            if len(members) > 0: centroids[c] = members.mean(axis=0)

    # 1. Silhouette Score
    s_scores = []
    for i in range(N):
        c_i = labels[i]
        same_cluster = X[(labels == c_i) & (np.arange(N) != i)]
        a_i = np.mean(np.linalg.norm(same_cluster - X[i], axis=1)) if len(same_cluster) > 0 else 0.0

        other_b = []
        for other_c in range(k):
            if other_c == c_i: continue
            other_members = X[labels == other_c]
            if len(other_members) > 0:
                other_b.append(np.mean(np.linalg.norm(other_members - X[i], axis=1)))
        b_i = min(other_b) if other_b else 0.0
        s_i = (b_i - a_i) / max(a_i, b_i) if max(a_i, b_i) > 1e-8 else 0.0
        s_scores.append(s_i)
    mean_sil = float(np.mean(s_scores))

    # 2. One-Way ANOVA on macro target
    grand_mean = np.mean(y_macro)
    ss_between = sum(len(y_macro[labels == c]) * (np.mean(y_macro[labels == c]) - grand_mean)**2 for c in range(k) if len(y_macro[labels == c]) > 0)
    ss_within = sum(np.sum((y_macro[labels == c] - np.mean(y_macro[labels == c]))**2) for c in range(k) if len(y_macro[labels == c]) > 0)

    df_between = k - 1
    df_within = N - k
    ms_between = ss_between / df_between if df_between > 0 else 0.0
    ms_within = ss_within / df_within if df_within > 0 else 1e-8
    f_stat = ms_between / ms_within if ms_within > 1e-8 else 0.0

    return mean_sil, float(f_stat), float(ss_between / (ss_between + ss_within + 1e-8))


# ==============================================================================
# 4. MASTER AUDIT CONTROLLER
# ==============================================================================

def run_milestone_9_3a_audit():
    harness = AuditHarness()
    corpus = get_audit_corpus()

    # Token check
    for item in corpus:
        _ = harness.model.to_single_token(item["correct"])
        _ = harness.model.to_single_token(item["distractor"])
    print(f"✅ Verified token integrity across {len(corpus)} evaluation tasks.")

    print("\n" + "=" * 135)
    print("🔬 MILESTONE 9.3A: DATASET EXTRACTION & READOUT SPECTRUM PROFILING (N=40 Tasks)")
    print("=" * 135)

    phi_list = []
    target_records = {
        "A1_ZeroAblate": {"dyn_margin": [], "target_logit": [], "sem_margin": [], "kl_div": [], "prob_shift": []},
        "A2_HeavyBoost": {"dyn_margin": [], "target_logit": [], "sem_margin": [], "kl_div": [], "prob_shift": []}
    }

    for item in corpus:
        phi, targets = harness.extract_fingerprint_and_targets(item)
        phi_list.append(phi)
        for act in ["A1_ZeroAblate", "A2_HeavyBoost"]:
            for m_key in target_records[act]:
                target_records[act][m_key].append(targets[act][m_key])

    X_phi = np.stack(phi_list)
    print(f"✅ Feature matrix constructed: {X_phi.shape[0]} episodes x {X_phi.shape[1]} micro-probe dimensions.")

    # -------------------------------------------------------------------------
    # AUDIT 1: PERMUTATION TEST (Real vs 500 Shuffled Baselines)
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("📊 AUDIT 1: PERMUTATION TEST (Real Phi vs. 500 Shuffled Baselines)")
    print("=" * 135)
    print(f"{'Macro Action':<16} | {'Target Metric':<20} | {'Real LOO-MAE':<14} | {'Shuffled Mean MAE':<18} | {'p-value (Significance)'}")
    print("-" * 135)

    permutation_results = {}
    n_perms = 500

    for act in ["A1_ZeroAblate", "A2_HeavyBoost"]:
        permutation_results[act] = {}
        for m_key in ["dyn_margin", "target_logit", "sem_margin", "kl_div", "prob_shift"]:
            y = np.array(target_records[act][m_key])
            real_mae = leave_one_out_ridge_mae(X_phi, y)

            shuffled_maes = []
            for _ in range(n_perms):
                y_shuff = np.random.permutation(y)
                shuffled_maes.append(leave_one_out_ridge_mae(X_phi, y_shuff))

            shuff_mean = float(np.mean(shuffled_maes))
            p_val = float((np.sum(np.array(shuffled_maes) <= real_mae) + 1) / (n_perms + 1))

            sig_str = "🟢 SIGNIFICANT (p < 0.05)" if p_val < 0.05 else "🔴 NOISE (p >= 0.05)"
            print(f"{act:<16} | {m_key:<20} | {real_mae:12.4f}   | {shuff_mean:16.4f}   | p = {p_val:.4f}  {sig_str}")

            permutation_results[act][m_key] = {
                "real_mae": real_mae,
                "shuffled_mean_mae": shuff_mean,
                "p_value": p_val,
                "is_significant": bool(p_val < 0.05)
            }

    # -------------------------------------------------------------------------
    # AUDIT 2: CAUSAL CLUSTER & REGIME VALIDITY (ANOVA & Silhouette)
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("📊 AUDIT 2: CAUSAL REGIME VALIDITY (Geometric Silhouette vs. Causal Outcome ANOVA)")
    print("=" * 135)
    print(f"{'K Clusters':<12} | {'Silhouette Score':<18} | {'F-Stat (ZeroAblate)':<22} | {'Variance Explained (η²)':<24} | {'Regime Reality Verdict'}")
    print("-" * 135)

    y_ablate = np.array(target_records["A1_ZeroAblate"]["target_logit"])
    cluster_results = {}

    for k in [2, 3, 4, 5]:
        sil, f_val, eta_sq = compute_silhouette_and_anova(X_phi, y_ablate, k=k)
        verdict = "🟢 REAL REGIMES" if (f_val > 3.0 and sil > 0.35) else "🔴 ARTIFACTUAL CLUSTERS"
        print(f"K = {k:<8} | {sil:16.4f}   | {f_val:20.4f}   | {eta_sq*100:20.1f}%   | {verdict}")
        cluster_results[f"K_{k}"] = {"silhouette": sil, "f_stat": f_val, "eta_sq": eta_sq}

    # -------------------------------------------------------------------------
    # AUDIT 3: TARGET READOUT SPECTRUM RANKING
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("📊 AUDIT 3: READOUT SPECTRUM PREDICTABILITY RANKING")
    print("=" * 135)
    print(f"{'Target Readout':<24} | {'ZeroAblate p-val':<18} | {'HeavyBoost p-val':<18} | {'Predictability Status'}")
    print("-" * 135)

    for m_key in ["dyn_margin", "target_logit", "sem_margin", "kl_div", "prob_shift"]:
        p1 = permutation_results["A1_ZeroAblate"][m_key]["p_value"]
        p2 = permutation_results["A2_HeavyBoost"][m_key]["p_value"]
        status = "🟢 HIGH SIGNAL READOUT" if (p1 < 0.05 or p2 < 0.05) else "🔴 UNSTABLE / NOISY READOUT"
        print(f"{m_key:<24} | p = {p1:12.4f}   | p = {p2:12.4f}   | {status}")

    print("=" * 135 + "\n")

    with open("m9_3a_audit_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "permutations": permutation_results,
            "clustering": cluster_results
        }, f, indent=2)
    print("📁 Comprehensive Audit Telemetry saved to: m9_3a_audit_results.json")


if __name__ == "__main__":
    run_milestone_9_3a_audit()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1721/460232359.py:33: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Verified token integrity across 40 evaluation tasks.

🔬 MILESTONE 9.3A: DATASET EXTRACTION & READOUT SPECTRUM PROFILING (N=40 Tasks)
✅ Feature matrix constructed: 40 episodes x 7 micro-probe dimensions.

📊 AUDIT 1: PERMUTATION TEST (Real Phi vs. 500 Shuffled Baselines)
Macro Action     | Target Metric        | Real LOO-MAE   | Shuffled Mean MAE  | p-value (Significance)
---------------------------------------------------------------------------------------------------------------------------------------
A1_ZeroAblate    | dyn_margin           |       0.2332   |           0.2494   | p = 0.0958  🔴 NOISE (p >= 0.05)
A1_ZeroAblate    | target_logit         |       0.3879   |           0.3957   | p = 0.2974  🔴 NOISE (p >= 0.05)
A1_ZeroAblate    | sem_margin           |       0.6927   |           0.6794   | p = 0.6607  🔴 NOISE (p >= 0.05)
A1_ZeroAblate    | kl_div               |       0.0367   |           0.0355   | p = 0.74

In [6]:
"""
m9_3b_dose_response_and_outcome_regimes.py
===========================================
Milestone 9.3B: Intervention Continuum & Outcome-First Causal Regime Discovery.

Key Protocol:
  1. 6D Orthogonal Micro-Fingerprint (Linearly Independent Probes).
  2. 7-Point Intervention Continuum (Dose-Response Curve: alpha in [0.0, 2.0]).
  3. Outcome-First Causal Clustering: Clusters response dynamics Y directly.
  4. Regime Predictability: Evaluates if Phi predicts the true causal regime.
  5. FDR Multi-Comparison Correction (Benjamini-Hochberg).
"""

import torch
import numpy as np
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC CONTINUUM HARNESS
# ==============================================================================

class ContinuumHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.d_head = self.model.cfg.d_head
        v_rand = torch.randn(self.d_head, dtype=torch.float32)
        self.v_ortho = (v_rand / v_rand.norm()).to(self.device)
        self.alpha_grid = [0.0, 0.25, 0.50, 0.75, 1.25, 1.50, 2.00]

    @torch.no_grad()
    def extract_phi_and_dose_response(self, prompt: str) -> Tuple[np.ndarray, np.ndarray, float]:
        tokens = self.model.to_tokens(prompt)
        nom_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

        top_ids = torch.topk(nom_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((nom_logits[t1] - nom_logits[t2]).item())

        # 6 Linearly Independent Micro-Probes
        # P1: L21H06 Scale-down (0.90)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P2: L21H06 Scale-up (1.10)
        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P3: L21H06 Ortho Jitter
        def p3_h6(act, hook): act[:, -1, 6, :] += 0.08 * act[:, -1, 6, :].norm() * self.v_ortho.to(act.dtype); return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h6)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P4: L21H01 Structural Scale-down (0.90)
        def p4_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p4_h1)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P5: L20 MLP Scale-down (0.90)
        def p5_mlp(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p5_mlp)]):
            d5 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P6: L17H01 Mid-layer Control Scale-down (0.90)
        def p6_h17(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.17.attn.hook_z", p6_h17)]):
            d6 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi = np.array([d1, d2, d3, d4, d5, d6], dtype=np.float32)

        # Measure Full Dose-Response Curve Y across alpha grid on L21H06
        dose_responses = []
        for alpha in self.alpha_grid:
            def dose_hook(act, hook): act[:, -1, 6, :] *= alpha; return act
            with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", dose_hook)]):
                m_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)
            d_alpha = float((m_logits[t1] - m_logits[t2]).item()) - base_margin
            dose_responses.append(d_alpha)

        y_curve = np.array(dose_responses, dtype=np.float32)
        return phi, y_curve, base_margin


# ==============================================================================
# 2. HOLDOUT CORPUS (48 BALANCED TASKS)
# ==============================================================================

def get_holdout_corpus():
    return [
        # Facts (12)
        {"id": "F01", "prompt": "The capital of France is the city of"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "F03", "prompt": "The official capital city of Germany is"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of"},
        {"id": "F05", "prompt": "The capital of Spain is the city of"},
        {"id": "F06", "prompt": "The capital of England is the city of"},
        {"id": "F07", "prompt": "The capital of Russia is the city of"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of"},
        {"id": "F09", "prompt": "The capital of Greece is the city of"},
        {"id": "F10", "prompt": "The capital of China is the city of"},
        {"id": "F11", "prompt": "The capital of Canada is the city of"},
        {"id": "F12", "prompt": "The capital of Australia is the city of"},
        # Bindings (12)
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "B07", "prompt": "William awarded a shiny medal to Oliver. The medal went to"},
        {"id": "B08", "prompt": "James threw the yellow frisbee to Lucas. The frisbee was caught by"},
        {"id": "B09", "prompt": "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by"},
        {"id": "B10", "prompt": "Henry presented an award to Ethan. The award was received by"},
        {"id": "B11", "prompt": "Samuel mailed a package to Alexander. The package arrived for"},
        {"id": "B12", "prompt": "George loaned some money to Noah. The money helped"},
        # Distractors / Interference (12)
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"},
        {"id": "D09", "prompt": "Alice gave a book to Bob, but Charlie immediately seized it. The book is with"},
        {"id": "D10", "prompt": "David sent a letter to Sarah, yet Emma intercepted the envelope. The letter reached"},
        {"id": "D11", "prompt": "Tom cooked dinner for Jack, but Oliver ate the entire plate. The food went to"},
        {"id": "D12", "prompt": "William wrote an article for Lisa, but Sophia published it. The author is"},
        # Negations & Counterfactuals (12)
        {"id": "N01", "prompt": "The capital of Spain is definitely not Rome, it is"},
        {"id": "N02", "prompt": "The official capital of Germany is not Munich, it is"},
        {"id": "N03", "prompt": "The capital of France is not Marseille, it is"},
        {"id": "N04", "prompt": "The capital of Japan is not Osaka, it is"},
        {"id": "N05", "prompt": "The capital of Italy is not Milan, it is"},
        {"id": "N06", "prompt": "The capital of Russia is not Saint Petersburg, it is"},
        {"id": "N07", "prompt": "In an alternate universe where France relocated its capital to Lyon, the capital of France is"},
        {"id": "N08", "prompt": "Suppose Japan officially changed its capital from Tokyo to Kyoto. The capital is"},
        {"id": "N09", "prompt": "If Germany renamed Munich as its capital city, the capital of Germany would be"},
        {"id": "N10", "prompt": "Imagine Italy established its seat of government in Venice. The capital of Italy is"},
        {"id": "N11", "prompt": "In a world where Spain moved its government to Barcelona, the Spanish capital is"},
        {"id": "N12", "prompt": "Assuming the UK declared Edinburgh as its sole capital, the UK capital is"}
    ]


# ==============================================================================
# 3. STATISTICAL & FDR ANALYSIS ENGINES
# ==============================================================================

def benjamini_hochberg(p_values: List[float], alpha: float = 0.05) -> Tuple[List[bool], float]:
    """Applies Benjamini-Hochberg FDR correction."""
    p_arr = np.array(p_values)
    n = len(p_arr)
    sorted_indices = np.argsort(p_arr)
    sorted_p = p_arr[sorted_indices]

    thresholds = (np.arange(1, n + 1) / n) * alpha
    significant = sorted_p <= thresholds

    if np.any(significant):
        max_sig_idx = np.max(np.where(significant)[0])
        p_crit = float(sorted_p[max_sig_idx])
    else:
        p_crit = 0.0

    orig_sig = p_arr <= p_crit
    return orig_sig.tolist(), p_crit


def leave_one_out_ridge_prediction(X: np.ndarray, y: np.ndarray, alpha: float = 0.1) -> Tuple[float, float]:
    N, D = X.shape
    errors = []
    preds = []
    for i in range(N):
        train_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_val, y_val = X[i:i+1], y[i]
        w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
        p = float(np.dot(X_val, w)[0])
        preds.append(p)
        errors.append(abs(p - y_val))

    corr = float(np.corrcoef(preds, y)[0, 1]) if np.std(preds) > 1e-6 else 0.0
    return float(np.mean(errors)), corr


# ==============================================================================
# 4. MASTER CONTROLLER
# ==============================================================================

def run_milestone_9_3b():
    harness = ContinuumHarness()
    corpus = get_holdout_corpus()
    print("\n" + "=" * 135)
    print(f"🔬 MILESTONE 9.3B: DOSE-RESPONSE CONTINUUM & OUTCOME-FIRST REGIME AUDIT (N={len(corpus)} Tasks)")
    print("=" * 135)

    phi_list, y_curves_list = [], []
    for item in corpus:
        phi, y_curve, _ = harness.extract_phi_and_dose_response(item["prompt"])
        phi_list.append(phi)
        y_curves_list.append(y_curve)

    X_phi = np.stack(phi_list)
    Y_curves = np.stack(y_curves_list)
    print(f"✅ Extracted 6D Micro-Fingerprints (Shape: {X_phi.shape}) & 7D Dose-Response Curves (Shape: {Y_curves.shape})")

    # -------------------------------------------------------------------------
    # PART 1: CONTINUOUS DOSE-RESPONSE PREDICTABILITY & PERMUTATION TEST
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("📊 PART 1: DOSE-RESPONSE CONTINUUM PREDICTABILITY (Permutation Tests on 7 Alpha Levels)")
    print("=" * 135)
    print(f"{'Alpha Level':<14} | {'LOO-MAE':<10} | {'Pearson ρ':<12} | {'Shuff MAE':<12} | {'Raw p-value':<14} | {'FDR Sig (q < 0.05)'}")
    print("-" * 135)

    raw_p_values = []
    point_results = []
    n_perms = 500

    for idx, alpha in enumerate(harness.alpha_grid):
        y_alpha = Y_curves[:, idx]
        real_mae, real_corr = leave_one_out_ridge_prediction(X_phi, y_alpha)

        shuff_maes = []
        for _ in range(n_perms):
            y_s = np.random.permutation(y_alpha)
            m_s, _ = leave_one_out_ridge_prediction(X_phi, y_s)
            shuff_maes.append(m_s)

        p_val = float((np.sum(np.array(shuff_maes) <= real_mae) + 1) / (n_perms + 1))
        raw_p_values.append(p_val)
        point_results.append({
            "alpha": alpha, "mae": real_mae, "corr": real_corr,
            "shuff_mae": float(np.mean(shuff_maes)), "p_val": p_val
        })

    # FDR Correction across all 7 continuum points
    fdr_sig_flags, p_threshold = benjamini_hochberg(raw_p_values, alpha=0.05)

    for idx, res in enumerate(point_results):
        sig_str = "🟢 SIGNIFICANT" if fdr_sig_flags[idx] else "🔴 NOT SIGNIFICANT"
        print(f"α = {res['alpha']:<8.2f} | {res['mae']:8.4f}   | {res['corr']:+10.4f}   | {res['shuff_mae']:10.4f}   | p = {res['p_val']:.4f}       | {sig_str}")

    # -------------------------------------------------------------------------
    # PART 2: OUTCOME-FIRST CAUSAL REGIME DISCOVERY (Clustering on Y)
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("📊 PART 2: OUTCOME-FIRST CAUSAL REGIME DISCOVERY (K-Means on Actual Response Dynamics Y)")
    print("=" * 135)

    # Cluster on Y directly to find true behavioral archetypes
    np.random.seed(SEED)
    k_regimes = 3
    init_centroids = Y_curves[np.random.choice(len(Y_curves), k_regimes, replace=False)].copy()
    for _ in range(25):
        dists = np.linalg.norm(Y_curves[:, None, :] - init_centroids[None, :, :], axis=-1)
        causal_labels = np.argmin(dists, axis=1)
        for c in range(k_regimes):
            mems = Y_curves[causal_labels == c]
            if len(mems) > 0: init_centroids[c] = mems.mean(axis=0)

    print("Causal Archetype Centroids (Dose-Response Profiles):")
    for c in range(k_regimes):
        profile_str = " ".join([f"{v:+.2f}" for v in init_centroids[c]])
        n_members = int(np.sum(causal_labels == c))
        print(f"  • Archetype #{c} (N={n_members:2d}): [{profile_str}]")

    # Evaluate if Phi can classify these outcome-first regimes (Leave-One-Out Nearest Centroid)
    correct_regime_preds = 0
    N = len(X_phi)
    for i in range(N):
        train_idx = [j for j in range(N) if j != i]
        X_tr, y_labels_tr = X_phi[train_idx], causal_labels[train_idx]

        # Centroids in Phi space for each true outcome regime
        phi_centroids = np.array([X_tr[y_labels_tr == c].mean(axis=0) if np.sum(y_labels_tr == c) > 0 else np.zeros(X_phi.shape[1]) for c in range(k_regimes)])
        pred_regime = int(np.argmin(np.linalg.norm(phi_centroids - X_phi[i:i+1], axis=1)))
        if pred_regime == causal_labels[i]:
            correct_regime_preds += 1

    regime_acc = (correct_regime_preds / N) * 100.0
    chance_acc = (1.0 / k_regimes) * 100.0

    print("-" * 135)
    print(f"Predicting True Causal Archetype from Micro-Fingerprint Φ(x):")
    print(f"  • Classification Accuracy : {regime_acc:.1f}% vs. Random Chance: {chance_acc:.1f}%")

    if regime_acc >= (chance_acc + 15.0):
        print("👉 REGIME AUDIT VERDICT: 🟢 OUTCOME_FIRST_REGIMES_PREDICTABLE (Φ contains causal archetype signature)")
    else:
        print("👉 REGIME AUDIT VERDICT: 🔴 REGIMES_NOT_PREDICTABLE_FROM_PHI")
    print("=" * 135 + "\n")

    with open("m9_3b_continuum_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "continuum": point_results,
            "fdr_threshold": p_threshold,
            "regime_accuracy": regime_acc
        }, f, indent=2)
    print("📁 Telemetry saved to: m9_3b_continuum_results.json")


if __name__ == "__main__":
    run_milestone_9_3b()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1721/1553956775.py:34: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 9.3B: DOSE-RESPONSE CONTINUUM & OUTCOME-FIRST REGIME AUDIT (N=48 Tasks)
✅ Extracted 6D Micro-Fingerprints (Shape: (48, 6)) & 7D Dose-Response Curves (Shape: (48, 7))

📊 PART 1: DOSE-RESPONSE CONTINUUM PREDICTABILITY (Permutation Tests on 7 Alpha Levels)
Alpha Level    | LOO-MAE    | Pearson ρ    | Shuff MAE    | Raw p-value    | FDR Sig (q < 0.05)
---------------------------------------------------------------------------------------------------------------------------------------
α = 0.00     |   0.2054   |    +0.4627   |     0.2448   | p = 0.0020       | 🟢 SIGNIFICANT
α = 0.25     |   0.1741   |    +0.5498   |     0.2036   | p = 0.0060       | 🟢 SIGNIFICANT
α = 0.50     |   0.1255   |    +0.5842   |     0.1516   | p = 0.0020       | 🟢 SIGNIFICANT
α = 0.75     |   0.0729   |    +0.6882   |     0.1040   | p = 0.0020       | 🟢 SIGNIFICANT
α = 1.25     |   0.0713   |    +0.5848   |     0.0875   | p = 0.0020    

In [7]:
"""
m9_4_cross_intervention_prediction.py
======================================
Milestone 9.4: Cross-Intervention & Cross-Component Generalization Benchmark.

Core Protocol:
  - Input Fingerprint Phi(x): Extracted ONLY from L21H06, L21H01, and L20_MLP (4 probes).
  - Target Macro-Interventions: 4 COMPLETELY UNSEEN target components:
      1. Y1: L21H02 Zero-Ablation (Unseen Intra-Layer Attention Head).
      2. Y2: L22H05 Zero-Ablation (Unseen Downstream Layer Head).
      3. Y3: L21_MLP 0.5x Damping (Unseen In-Layer MLP Block).
      4. Y4: L18H03 2.0x Heavy Boost (Unseen Upstream Attention Head).
  - Statistical Rigor: LOO-Ridge, 500 Permutations per target, FDR (q < 0.05), vs. Passive Baseline.
"""

import torch
import numpy as np
import json
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC HARNESS
# ==============================================================================

class CrossInterventionHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()

    @torch.no_grad()
    def extract_phi_passive_and_targets(self, prompt: str) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
        tokens = self.model.to_tokens(prompt)
        nom_logits, cache = self.model.run_with_cache(tokens)
        last_logits = nom_logits[0, -1, :].detach().cpu().to(torch.float32)

        top_ids = torch.topk(last_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((last_logits[t1] - last_logits[t2]).item())

        # Passive State: Normalized resid_pre at Layer 21
        resid_21 = cache["blocks.21.hook_resid_pre"][0, -1, :].detach().cpu().to(torch.float32).numpy()
        passive_vec = resid_21 / (np.linalg.norm(resid_21) + 1e-8)

        # Fixed Input Micro-Probes (Phi): ONLY L21H06, L21H01, L20MLP
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p3_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h1)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p4_mlp20(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p4_mlp20)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi = np.array([d1, d2, d3, d4], dtype=np.float32)

        # Completely Unseen Target Macro-Interventions (Y)
        unseen_macro_deltas = {}

        # Target Y1: L21H02 Zero-Ablate (Intra-layer unseen head)
        def y1_hook(act, hook): act[:, -1, 2, :] *= 0.0; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", y1_hook)]):
            unseen_macro_deltas["Y1_L21H02_Ablate"] = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # Target Y2: L22H05 Zero-Ablate (Downstream layer head)
        def y2_hook(act, hook): act[:, -1, 5, :] *= 0.0; return act
        with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", y2_hook)]):
            unseen_macro_deltas["Y2_L22H05_Ablate"] = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # Target Y3: L21_MLP 0.5x Damping (In-layer MLP block)
        def y3_hook(act, hook): act[:, -1, :] *= 0.50; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.hook_mlp_out", y3_hook)]):
            unseen_macro_deltas["Y3_L21MLP_Damp"] = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # Target Y4: L18H03 2.0x Boost (Upstream attention head)
        def y4_hook(act, hook): act[:, -1, 3, :] *= 2.0; return act
        with self.model.hooks(fwd_hooks=[("blocks.18.attn.hook_z", y4_hook)]):
            unseen_macro_deltas["Y4_L18H03_Boost"] = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        return phi, passive_vec, unseen_macro_deltas


# ==============================================================================
# 2. EVALUATION CORPUS (48 BALANCED TASKS)
# ==============================================================================

def get_balanced_corpus():
    return [
        # Facts (12)
        {"id": "F01", "prompt": "The capital of France is the city of"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "F03", "prompt": "The official capital city of Germany is"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of"},
        {"id": "F05", "prompt": "The capital of Spain is the city of"},
        {"id": "F06", "prompt": "The capital of England is the city of"},
        {"id": "F07", "prompt": "The capital of Russia is the city of"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of"},
        {"id": "F09", "prompt": "The capital of Greece is the city of"},
        {"id": "F10", "prompt": "The capital of China is the city of"},
        {"id": "F11", "prompt": "The capital of Canada is the city of"},
        {"id": "F12", "prompt": "The capital of Australia is the city of"},
        # Bindings (12)
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "B07", "prompt": "William awarded a shiny medal to Oliver. The medal went to"},
        {"id": "B08", "prompt": "James threw the yellow frisbee to Lucas. The frisbee was caught by"},
        {"id": "B09", "prompt": "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by"},
        {"id": "B10", "prompt": "Henry presented an award to Ethan. The award was received by"},
        {"id": "B11", "prompt": "Samuel mailed a package to Alexander. The package arrived for"},
        {"id": "B12", "prompt": "George loaned some money to Noah. The money helped"},
        # Distractors (12)
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"},
        {"id": "D09", "prompt": "Alice gave a book to Bob, but Charlie immediately seized it. The book is with"},
        {"id": "D10", "prompt": "David sent a letter to Sarah, yet Emma intercepted the envelope. The letter reached"},
        {"id": "D11", "prompt": "Tom cooked dinner for Jack, but Oliver ate the entire plate. The food went to"},
        {"id": "D12", "prompt": "William wrote an article for Lisa, but Sophia published it. The author is"},
        # Counterfactuals & Negations (12)
        {"id": "N01", "prompt": "The capital of Spain is definitely not Rome, it is"},
        {"id": "N02", "prompt": "The official capital of Germany is not Munich, it is"},
        {"id": "N03", "prompt": "The capital of France is not Marseille, it is"},
        {"id": "N04", "prompt": "The capital of Japan is not Osaka, it is"},
        {"id": "N05", "prompt": "The capital of Italy is not Milan, it is"},
        {"id": "N06", "prompt": "The capital of Russia is not Saint Petersburg, it is"},
        {"id": "N07", "prompt": "In an alternate universe where France relocated its capital to Lyon, the capital of France is"},
        {"id": "N08", "prompt": "Suppose Japan officially changed its capital from Tokyo to Kyoto. The capital is"},
        {"id": "N09", "prompt": "If Germany renamed Munich as its capital city, the capital of Germany would be"},
        {"id": "N10", "prompt": "Imagine Italy established its seat of government in Venice. The capital of Italy is"},
        {"id": "N11", "prompt": "In a world where Spain moved its government to Barcelona, the Spanish capital is"},
        {"id": "N12", "prompt": "Assuming the UK declared Edinburgh as its sole capital, the UK capital is"}
    ]


# ==============================================================================
# 3. STATISTICAL EVALUATION ENGINE
# ==============================================================================

def leave_one_out_eval(X: np.ndarray, y: np.ndarray, alpha: float = 0.1) -> Tuple[float, float]:
    N, D = X.shape
    preds, errs = [], []
    for i in range(N):
        tr_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_val, y_val = X[i:i+1], y[i]
        w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
        p = float(np.dot(X_val, w)[0])
        preds.append(p)
        errs.append(abs(p - y_val))
    corr = float(np.corrcoef(preds, y)[0, 1]) if np.std(preds) > 1e-6 else 0.0
    return float(np.mean(errs)), corr


def apply_fdr(p_values: List[float], alpha: float = 0.05) -> List[bool]:
    p_arr = np.array(p_values)
    n = len(p_arr)
    sorted_idx = np.argsort(p_arr)
    sorted_p = p_arr[sorted_idx]
    thresholds = (np.arange(1, n + 1) / n) * alpha
    sig = sorted_p <= thresholds
    p_crit = float(sorted_p[np.max(np.where(sig)[0])]) if np.any(sig) else 0.0
    return (p_arr <= p_crit).tolist()


# ==============================================================================
# 4. MASTER RUNNER
# ==============================================================================

def run_milestone_9_4():
    harness = CrossInterventionHarness()
    corpus = get_balanced_corpus()

    print("\n" + "=" * 140)
    print(f"🔬 MILESTONE 9.4: CROSS-INTERVENTION & CROSS-COMPONENT GENERALIZATION BENCHMARK (N={len(corpus)} Tasks)")
    print("=" * 140)

    phi_list, pass_list = [], []
    macro_targets = {"Y1_L21H02_Ablate": [], "Y2_L22H05_Ablate": [], "Y3_L21MLP_Damp": [], "Y4_L18H03_Boost": []}

    for item in corpus:
        phi, pass_vec, t_deltas = harness.extract_phi_passive_and_targets(item["prompt"])
        phi_list.append(phi)
        pass_list.append(pass_vec)
        for k in macro_targets:
            macro_targets[k].append(t_deltas[k])

    X_phi = np.stack(phi_list)
    X_pass = np.stack(pass_list)
    print(f"✅ Extracted 4D Fixed Probe Fingerprints (Shape: {X_phi.shape}) & 4 Unseen Macro Target Channels.")

    # -------------------------------------------------------------------------
    # STATISTICAL EVALUATION ACROSS UNSEEN TARGETS
    # -------------------------------------------------------------------------
    print("\n" + "=" * 140)
    print("📊 CROSS-COMPONENT TRANSFER PERFORMANCE & PERMUTATION SIGNIFICANCE")
    print("=" * 140)
    print(f"{'Target Action (Unseen)':<24} | {'Phi MAE':<10} | {'Phi ρ':<10} | {'Passive MAE':<12} | {'Global Mean MAE':<16} | {'Perm p-val':<12} | {'FDR Sig (q < 0.05)'}")
    print("-" * 140)

    raw_p_values = []
    target_results = []
    n_perms = 500

    for a_key in ["Y1_L21H02_Ablate", "Y2_L22H05_Ablate", "Y3_L21MLP_Damp", "Y4_L18H03_Boost"]:
        y = np.array(macro_targets[a_key])

        # Active Phi LOO
        mae_phi, corr_phi = leave_one_out_eval(X_phi, y)
        # Passive State LOO
        mae_pass, _ = leave_one_out_eval(X_pass, y, alpha=1.0)
        # Global Mean Baseline
        mae_mean = float(np.mean(np.abs(y - np.mean(y))))

        # Permutation Test on Phi
        shuff_maes = []
        for _ in range(n_perms):
            y_s = np.random.permutation(y)
            m_s, _ = leave_one_out_eval(X_phi, y_s)
            shuff_maes.append(m_s)

        p_val = float((np.sum(np.array(shuff_maes) <= mae_phi) + 1) / (n_perms + 1))
        raw_p_values.append(p_val)

        target_results.append({
            "target": a_key,
            "mae_phi": mae_phi,
            "corr_phi": corr_phi,
            "mae_pass": mae_pass,
            "mae_mean": mae_mean,
            "p_val": p_val
        })

    fdr_flags = apply_fdr(raw_p_values, alpha=0.05)

    for idx, res in enumerate(target_results):
        sig_str = "🟢 SIGNIFICANT" if fdr_flags[idx] else "🔴 NOT SIGNIFICANT"
        print(f"{res['target']:<24} | {res['mae_phi']:8.4f}   | {res['corr_phi']:+8.4f}   | {res['mae_pass']:10.4f}   | {res['mae_mean']:14.4f}   | p = {res['p_val']:.4f}     | {sig_str}")

    print("-" * 140)
    sig_count = sum(fdr_flags)
    mean_corr = float(np.mean([r["corr_phi"] for r in target_results]))

    if sig_count >= 2 and mean_corr >= 0.35:
        print("👉 BENCHMARK VERDICT: 🟢 CROSS_INTERVENTION_GENERALIZATION_VALIDATED")
        print("   (Micro-fingerprints from base circuit reliably predict causal consequences on unseen heads/layers)")
    else:
        print("👉 BENCHMARK VERDICT: 🔴 LOCAL_COORDINATE_BOUND (Limited to base components)")
    print("=" * 140 + "\n")

    with open("m9_4_cross_intervention_results.json", "w", encoding="utf-8") as f:
        json.dump({"results": target_results, "fdr_flags": fdr_flags}, f, indent=2)
    print("📁 Telemetry saved to: m9_4_cross_intervention_results.json")


if __name__ == "__main__":
    run_milestone_9_4()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1721/1986997814.py:35: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 9.4: CROSS-INTERVENTION & CROSS-COMPONENT GENERALIZATION BENCHMARK (N=48 Tasks)
✅ Extracted 4D Fixed Probe Fingerprints (Shape: (48, 4)) & 4 Unseen Macro Target Channels.

📊 CROSS-COMPONENT TRANSFER PERFORMANCE & PERMUTATION SIGNIFICANCE
Target Action (Unseen)   | Phi MAE    | Phi ρ      | Passive MAE  | Global Mean MAE  | Perm p-val   | FDR Sig (q < 0.05)
--------------------------------------------------------------------------------------------------------------------------------------------
Y1_L21H02_Ablate         |   0.0787   |  +0.2409   |     0.0823   |         0.0807   | p = 0.1118     | 🔴 NOT SIGNIFICANT
Y2_L22H05_Ablate         |   0.0698   |  +0.5482   |     0.0794   |         0.0729   | p = 0.0020     | 🟢 SIGNIFICANT
Y3_L21MLP_Damp           |   0.3855   |  +0.5558   |     0.5203   |         0.4974   | p = 0.0020     | 🟢 SIGNIFICANT
Y4_L18H03_Boost          |   0.1184   |  +0.2080   |     0.1074 

In [8]:
"""
m9_5_intervention_conditioned_self_model.py
============================================
Milestone 9.5: Intervention-Conditioned Continuous Causal Self-Model.

Core Objectives:
  1. Learns a unified mapping: f(Phi(x), e_Intervention) -> Delta Margin.
  2. Evaluates across a 10-action intervention grid on 48 balanced tasks (480 episodes).
  3. Benchmarks under strict Leave-One-Component-Out (LOCO) and Leave-One-Scale-Out (LOSO).
  4. Compares against:
     - Global Mean Baseline
     - Action-Specific Static Mean (ignores Phi context)
     - Passive State Model (resid_pre)
"""

import torch
import numpy as np
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. INTERVENTION DESCRIPTOR & ACTION SUITE
# ==============================================================================

@dataclass
class CausalAction:
    action_id: str
    target_name: str
    target_type: str   # "HEAD" or "MLP"
    layer: int
    head: int          # 0 for MLP
    scale: float       # alpha parameter (e.g., 0.0, 0.5, 1.5, 2.0)

    def get_descriptor(self) -> np.ndarray:
        """Encodes intervention parameters into a standardized continuous feature vector."""
        is_mlp = 1.0 if self.target_type == "MLP" else 0.0
        layer_norm = float(self.layer) / 28.0
        head_norm = float(self.head) / 16.0 if self.target_type == "HEAD" else 0.0
        scale_norm = self.scale / 2.0
        delta_scale = self.scale - 1.0  # Deviation from baseline (0.0 = identity)
        return np.array([layer_norm, is_mlp, head_norm, scale_norm, delta_scale], dtype=np.float32)


def get_macro_action_suite() -> List[CausalAction]:
    return [
        # Base Circuit Interventions (Seen in Training Pool)
        CausalAction("ACT_01_L21H06_Zero",  "L21H06", "HEAD", 21, 6, 0.0),
        CausalAction("ACT_02_L21H06_Damp",  "L21H06", "HEAD", 21, 6, 0.5),
        CausalAction("ACT_03_L21H06_Boost", "L21H06", "HEAD", 21, 6, 1.5),
        CausalAction("ACT_04_L21H06_Heavy", "L21H06", "HEAD", 21, 6, 2.0),
        CausalAction("ACT_05_L20MLP_Damp",  "L20MLP", "MLP",  20, 0, 0.5),
        CausalAction("ACT_06_L20MLP_Boost", "L20MLP", "MLP",  20, 0, 1.5),

        # Held-Out Target Components (For Cross-Component Testing)
        CausalAction("ACT_07_L21MLP_Damp",  "L21MLP", "MLP",  21, 0, 0.5),
        CausalAction("ACT_08_L21MLP_Zero",  "L21MLP", "MLP",  21, 0, 0.0),
        CausalAction("ACT_09_L22H05_Zero",  "L22H05", "HEAD", 22, 5, 0.0),
        CausalAction("ACT_10_L22H05_Boost", "L22H05", "HEAD", 22, 5, 1.5),
    ]


# ==============================================================================
# 2. CONTINUOUS SELF-MODEL DYNAMICS ENGINE
# ==============================================================================

class UnifiedSelfModel:
    def __init__(self, alpha_reg: float = 0.10):
        self.alpha_reg = alpha_reg
        self.w_joint = None

    def _build_joint_features(self, Phi: np.ndarray, e_I: np.ndarray) -> np.ndarray:
        """Constructs bilinear outer-product interaction features between Phi and Action descriptor."""
        outer = np.outer(Phi, e_I).flatten()
        bias_term = np.array([1.0], dtype=np.float32)
        return np.concatenate([outer, Phi, e_I, bias_term])

    def fit(self, episodes: List[Dict]):
        X = np.stack([self._build_joint_features(e["phi"], e["action"].get_descriptor()) for e in episodes])
        y = np.array([e["actual_delta"] for e in episodes], dtype=np.float32)
        D = X.shape[1]
        self.w_joint = np.linalg.solve(X.T @ X + self.alpha_reg * np.eye(D), X.T @ y)

    def predict(self, phi: np.ndarray, action: CausalAction) -> float:
        z = self._build_joint_features(phi, action.get_descriptor())
        return float(np.dot(z, self.w_joint))


# ==============================================================================
# 3. MECHANISTIC HARNESS
# ==============================================================================

class M9_5_Harness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()

    @torch.no_grad()
    def extract_phi_and_passive(self, prompt: str) -> Tuple[int, int, float, np.ndarray, np.ndarray]:
        tokens = self.model.to_tokens(prompt)
        nom_logits, cache = self.model.run_with_cache(tokens)
        last_logits = nom_logits[0, -1, :].detach().cpu().to(torch.float32)

        top_ids = torch.topk(last_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((last_logits[t1] - last_logits[t2]).item())

        # Passive State Vector: Normalized resid_pre Layer 21
        resid_21 = cache["blocks.21.hook_resid_pre"][0, -1, :].detach().cpu().to(torch.float32).numpy()
        passive_vec = resid_21 / (np.linalg.norm(resid_21) + 1e-8)

        # 4D Micro-Fingerprint (Phi)
        # P1: L21H06 (0.90 Scale)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P2: L21H06 (1.10 Scale)
        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P3: L21H01 (0.90 Scale)
        def p3_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h1)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # P4: L20MLP (0.90 Scale)
        def p4_mlp20(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p4_mlp20)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi = np.array([d1, d2, d3, d4], dtype=np.float32)
        return t1, t2, base_margin, phi, passive_vec

    @torch.no_grad()
    def measure_action_consequence(self, prompt: str, t1: int, t2: int, base_margin: float, action: CausalAction) -> float:
        tokens = self.model.to_tokens(prompt)

        if action.target_type == "HEAD":
            def hook_fn(act, hook): act[:, -1, action.head, :] *= action.scale; return act
            hook_point = f"blocks.{action.layer}.attn.hook_z"
        else:
            def hook_fn(act, hook): act[:, -1, :] *= action.scale; return act
            hook_point = f"blocks.{action.layer}.hook_mlp_out"

        with self.model.hooks(fwd_hooks=[(hook_point, hook_fn)]):
            mod_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

        actual_margin = float((mod_logits[t1] - mod_logits[t2]).item())
        return float(actual_margin - base_margin)


# ==============================================================================
# 4. CORPUS GENERATION (48 BALANCED PROMPTS)
# ==============================================================================

def get_full_corpus():
    return [
        # Facts (12)
        {"id": "F01", "prompt": "The capital of France is the city of"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "F03", "prompt": "The official capital city of Germany is"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of"},
        {"id": "F05", "prompt": "The capital of Spain is the city of"},
        {"id": "F06", "prompt": "The capital of England is the city of"},
        {"id": "F07", "prompt": "The capital of Russia is the city of"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of"},
        {"id": "F09", "prompt": "The capital of Greece is the city of"},
        {"id": "F10", "prompt": "The capital of China is the city of"},
        {"id": "F11", "prompt": "The capital of Canada is the city of"},
        {"id": "F12", "prompt": "The capital of Australia is the city of"},
        # Bindings (12)
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "B07", "prompt": "William awarded a shiny medal to Oliver. The medal went to"},
        {"id": "B08", "prompt": "James threw the yellow frisbee to Lucas. The frisbee was caught by"},
        {"id": "B09", "prompt": "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by"},
        {"id": "B10", "prompt": "Henry presented an award to Ethan. The award was received by"},
        {"id": "B11", "prompt": "Samuel mailed a package to Alexander. The package arrived for"},
        {"id": "B12", "prompt": "George loaned some money to Noah. The money helped"},
        # Distractors (12)
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"},
        {"id": "D09", "prompt": "Alice gave a book to Bob, but Charlie immediately seized it. The book is with"},
        {"id": "D10", "prompt": "David sent a letter to Sarah, yet Emma intercepted the envelope. The letter reached"},
        {"id": "D11", "prompt": "Tom cooked dinner for Jack, but Oliver ate the entire plate. The food went to"},
        {"id": "D12", "prompt": "William wrote an article for Lisa, but Sophia published it. The author is"},
        # Counterfactuals & Negations (12)
        {"id": "N01", "prompt": "The capital of Spain is definitely not Rome, it is"},
        {"id": "N02", "prompt": "The official capital of Germany is not Munich, it is"},
        {"id": "N03", "prompt": "The capital of France is not Marseille, it is"},
        {"id": "N04", "prompt": "The capital of Japan is not Osaka, it is"},
        {"id": "N05", "prompt": "The capital of Italy is not Milan, it is"},
        {"id": "N06", "prompt": "The capital of Russia is not Saint Petersburg, it is"},
        {"id": "N07", "prompt": "In an alternate universe where France relocated its capital to Lyon, the capital of France is"},
        {"id": "N08", "prompt": "Suppose Japan officially changed its capital from Tokyo to Kyoto. The capital is"},
        {"id": "N09", "prompt": "If Germany renamed Munich as its capital city, the capital of Germany would be"},
        {"id": "N10", "prompt": "Imagine Italy established its seat of government in Venice. The capital of Italy is"},
        {"id": "N11", "prompt": "In a world where Spain moved its government to Barcelona, the Spanish capital is"},
        {"id": "N12", "prompt": "Assuming the UK declared Edinburgh as its sole capital, the UK capital is"}
    ]


# ==============================================================================
# 5. MASTER BENCHMARK RUNNER
# ==============================================================================

def run_milestone_9_5():
    harness = M9_5_Harness()
    corpus = get_full_corpus()
    actions = get_macro_action_suite()

    print("\n" + "=" * 140)
    print(f"🔬 MILESTONE 9.5: INTERVENTION-CONDITIONED CONTINUOUS CAUSAL SELF-MODEL BENCHMARK")
    print(f"   Corpus: {len(corpus)} Tasks | Action Space: {len(actions)} Interventions | Total Episodes: {len(corpus)*len(actions)}")
    print("=" * 140)

    # 1. Experience Collection
    dataset = []
    print("⏳ Collecting ground-truth causal experience across full intervention grid...")
    for item in corpus:
        t1, t2, base_m, phi, pass_v = harness.extract_phi_and_passive(item["prompt"])
        for act in actions:
            actual_d = harness.measure_action_consequence(item["prompt"], t1, t2, base_m, act)
            dataset.append({
                "task_id": item["id"],
                "prompt": item["prompt"],
                "phi": phi,
                "passive_vec": pass_v,
                "action": act,
                "actual_delta": actual_d
            })
    print(f"✅ Experience database populated with {len(dataset)} empirical transition episodes.")

    # -------------------------------------------------------------------------
    # EXPERIMENT 1: LEAVE-ONE-COMPONENT-OUT (LOCO) GENERALIZATION
    # -------------------------------------------------------------------------
    print("\n" + "=" * 140)
    print("📊 EXPERIMENT 1: LEAVE-ONE-COMPONENT-OUT (LOCO) CROSS-COMPONENT EVALUATION")
    print("=" * 140)
    print(f"{'Held-Out Component':<22} | {'Self-Model MAE':<16} | {'Action-Mean MAE':<18} | {'Passive MAE':<14} | {'Self-Model ρ':<14} | {'Advantage over Static'}")
    print("-" * 140)

    unique_components = sorted(list(set(a.target_name for a in actions)))
    loco_results = []

    for comp in unique_components:
        train_eps = [e for e in dataset if e["action"].target_name != comp]
        test_eps = [e for e in dataset if e["action"].target_name == comp]

        # Fit Self-Model on remaining components
        model = UnifiedSelfModel(alpha_reg=0.10)
        model.fit(train_eps)

        # Passive Model (Baseline)
        X_pass_tr = np.stack([np.concatenate([np.outer(e["passive_vec"][:16], e["action"].get_descriptor()).flatten(), e["action"].get_descriptor()]) for e in train_eps])
        y_tr = np.array([e["actual_delta"] for e in train_eps])
        w_pass = np.linalg.solve(X_pass_tr.T @ X_pass_tr + 1.0 * np.eye(X_pass_tr.shape[1]), X_pass_tr.T @ y_tr)

        # Evaluate on Held-out Component
        y_actual, y_pred_self, y_pred_pass, y_pred_act_mean = [], [], [], []
        action_means = {a.action_id: float(np.mean([e["actual_delta"] for e in train_eps if e["action"].action_id == a.action_id])) for a in actions}

        for e in test_eps:
            p_self = model.predict(e["phi"], e["action"])
            z_pass = np.concatenate([np.outer(e["passive_vec"][:16], e["action"].get_descriptor()).flatten(), e["action"].get_descriptor()])
            p_pass = float(np.dot(z_pass, w_pass))

            # If action was never seen in training, fallback to global mean
            act_m = action_means.get(e["action"].action_id, float(np.mean(y_tr)))
            if np.isnan(act_m): act_m = float(np.mean(y_tr))

            y_actual.append(e["actual_delta"])
            y_pred_self.append(p_self)
            y_pred_pass.append(p_pass)
            y_pred_act_mean.append(act_m)

        mae_self = float(np.mean(np.abs(np.array(y_pred_self) - np.array(y_actual))))
        mae_pass = float(np.mean(np.abs(np.array(y_pred_pass) - np.array(y_actual))))
        mae_act_m = float(np.mean(np.abs(np.array(y_pred_act_mean) - np.array(y_actual))))

        corr_self = float(np.corrcoef(y_pred_self, y_actual)[0, 1]) if np.std(y_pred_self) > 1e-6 else 0.0
        adv_pct = ((mae_act_m - mae_self) / (mae_act_m + 1e-8)) * 100.0

        print(f"{comp:<22} | {mae_self:14.4f}   | {mae_act_m:16.4f}   | {mae_pass:12.4f}   | {corr_self:+12.4f}   | {adv_pct:+18.1f}%")
        loco_results.append({
            "component": comp, "mae_self": mae_self, "mae_act_mean": mae_act_m,
            "mae_pass": mae_pass, "corr_self": corr_self, "adv_pct": adv_pct
        })

    # -------------------------------------------------------------------------
    # EXPERIMENT 2: LEAVE-ONE-SCALE-OUT (LOSO) EXTRAPOLATION EVALUATION
    # -------------------------------------------------------------------------
    print("\n" + "=" * 140)
    print("📊 EXPERIMENT 2: LEAVE-ONE-SCALE-OUT (LOSO) EXTREME DOSE EXTRAPOLATION")
    print("=" * 140)
    print(f"{'Held-Out Scale (Alpha)':<22} | {'Self-Model MAE':<16} | {'Action-Mean MAE':<18} | {'Self-Model ρ':<14} | {'Generalization Status'}")
    print("-" * 140)

    unique_scales = sorted(list(set(a.scale for a in actions)))
    loso_results = []

    for sc in unique_scales:
        train_eps = [e for e in dataset if e["action"].scale != sc]
        test_eps = [e for e in dataset if e["action"].scale == sc]

        model = UnifiedSelfModel(alpha_reg=0.10)
        model.fit(train_eps)

        y_actual = [e["actual_delta"] for e in test_eps]
        y_pred = [model.predict(e["phi"], e["action"]) for e in test_eps]
        y_global = [float(np.mean([x["actual_delta"] for x in train_eps])) for _ in test_eps]

        mae_self = float(np.mean(np.abs(np.array(y_pred) - np.array(y_actual))))
        mae_glob = float(np.mean(np.abs(np.array(y_global) - np.array(y_actual))))
        corr_self = float(np.corrcoef(y_pred, y_actual)[0, 1]) if np.std(y_pred) > 1e-6 else 0.0

        status = "🟢 VALIDATED" if (corr_self >= 0.40 and mae_self < mae_glob) else "🔴 LIMITED EXTRAPOLATION"
        print(f"α = {sc:<18.2f} | {mae_self:14.4f}   | {mae_glob:16.4f}   | {corr_self:+12.4f}   | {status}")
        loso_results.append({"scale": sc, "mae_self": mae_self, "mae_glob": mae_glob, "corr_self": corr_self})

    # -------------------------------------------------------------------------
    # SUMMARY AUDIT & VERDICT
    # -------------------------------------------------------------------------
    print("\n" + "=" * 140)
    print("📊 MILESTONE 9.5 FINAL SYNTHESIS & BENCHMARK AUDIT")
    print("=" * 140)
    mean_loco_adv = float(np.mean([r["adv_pct"] for r in loco_results]))
    mean_loco_corr = float(np.mean([r["corr_self"] for r in loco_results]))
    mean_loso_corr = float(np.mean([r["corr_self"] for r in loso_results]))

    print(f"  • Average Advantage over Static Action Lookup (LOCO) : {mean_loco_adv:+.1f}%")
    print(f"  • Mean Cross-Component Correlation (LOCO)           : {mean_loco_corr:+.4f}")
    print(f"  • Mean Dose-Extrapolation Correlation (LOSO)         : {mean_loso_corr:+.4f}")
    print("-" * 140)

    if mean_loco_adv >= 10.0 and mean_loco_corr >= 0.35 and mean_loso_corr >= 0.40:
        print("👉 FINAL AUDIT VERDICT: 🟢 UNIFIED_PREDICTIVE_SELF_MODEL_VALIDATED")
        print("   (Continuous self-model successfully predicts causal outcomes conditioned on internal state and action descriptors)")
    else:
        print("👉 FINAL AUDIT VERDICT: 🔴 PARTIAL_CONDITIONAL_GENERALIZATION")
    print("=" * 140 + "\n")

    with open("m9_5_unified_self_model_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "loco_results": loco_results,
            "loso_results": loso_results,
            "metrics": {
                "mean_loco_adv": mean_loco_adv,
                "mean_loco_corr": mean_loco_corr,
                "mean_loso_corr": mean_loso_corr
            }
        }, f, indent=2)
    print("📁 Comprehensive Telemetry saved to: m9_5_unified_self_model_results.json")


if __name__ == "__main__":
    run_milestone_9_5()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1721/1085621600.py:103: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 9.5: INTERVENTION-CONDITIONED CONTINUOUS CAUSAL SELF-MODEL BENCHMARK
   Corpus: 48 Tasks | Action Space: 10 Interventions | Total Episodes: 480
⏳ Collecting ground-truth causal experience across full intervention grid...
✅ Experience database populated with 480 empirical transition episodes.

📊 EXPERIMENT 1: LEAVE-ONE-COMPONENT-OUT (LOCO) CROSS-COMPONENT EVALUATION
Held-Out Component     | Self-Model MAE   | Action-Mean MAE    | Passive MAE    | Self-Model ρ   | Advantage over Static
--------------------------------------------------------------------------------------------------------------------------------------------
L20MLP                 |         0.4377   |           0.4039   |       0.4081   |      +0.1760   |               -8.4%
L21H06                 |         0.2816   |           0.2048   |       0.2175   |      +0.2307   |              -37.5%
L21MLP                 |         0.6512   |           

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
